In [10]:
!pip install streamlit pyngrok -q

In [11]:
%%writefile dashboard.py
"""
================================================================================
 REAL-TIME ANOMALY DETECTION DASHBOARD  (v3 — Severity-Based Rule Engine)
 Immersion Aluminium Holding Furnace — 1120 kg/ch
 PT. Astra Honda Motor  |  Casting Operations — Predictive Maintenance
================================================================================

Pipeline yang direplikasi di dashboard ini (harus sinkron dengan Notebook 1-3):
    Notebook 1 (cleansing)   -> cleaned_data.csv   -> direplikasi di cleanse_raw_data()
    Notebook 2 (processing)  -> labeled_data.csv   -> direplikasi di engineer_batch_features()
                                                        + apply_layer1_rules_batch()
    Notebook 3 (training)    -> model_ocsvm.pkl, model_lof.pkl, model_xgb.pkl,
                                 scaler.pkl, model_metadata.json
    Notebook 4 (dashboard)   -> file ini (dashboard.py)

--------------------------------------------------------------------------------
CARA MENJALANKAN DI KAGGLE NOTEBOOK
--------------------------------------------------------------------------------
Kaggle tidak mengizinkan port publik langsung, sehingga Streamlit harus dijalankan
sebagai proses background lalu di-tunnel keluar. Ada 2 opsi (pilih salah satu):

OPSI A — localtunnel (npx, tidak perlu akun/token):
    !pip install streamlit -q
    !npm install -g localtunnel -q
    import subprocess, time
    subprocess.Popen(["streamlit", "run", "dashboard.py",
                       "--server.port", "8501",
                       "--server.headless", "true",
                       "--server.enableCORS", "false"])
    time.sleep(8)
    # Password tunnel = IP publik Kaggle (ditampilkan otomatis oleh perintah di bawah)
    !curl -s https://loca.lt/mytunnelpassword
    !npx localtunnel --port 8501

OPSI B — pyngrok (butuh auth token dari https://dashboard.ngrok.com):
    !pip install streamlit pyngrok -q
    import subprocess, time
    from pyngrok import ngrok
    ngrok.set_auth_token("MASUKKAN_TOKEN_NGROK_ANDA")
    subprocess.Popen(["streamlit", "run", "dashboard.py",
                       "--server.port", "8501",
                       "--server.headless", "true"])
    time.sleep(8)
    public_url = ngrok.connect(8501)
    print("Dashboard tersedia di:", public_url)

Catatan:
  - Simpan cell di atas TERPISAH dari file dashboard.py ini (jalankan lewat %%writefile
    dashboard.py lalu jalankan cell subprocess pada cell berikutnya).
  - Jika MODEL_PATH / DATA_PATH di bawah tidak ditemukan, dashboard otomatis masuk ke
    "Mode Demo" (data + model sintetis) supaya tetap bisa langsung dijalankan/diuji
    di luar Kaggle tanpa error.
  - v3 CHANGELOG (sinkron Notebook 1-3 terbaru):
      1) molten_temp == 0 sekarang ditangani sebagai SENSOR FAULT (bukan temperature
         drop) lewat kolom molten_temp_valid / molten_temp_sensor_fault_flag.
      2) Layer-1 Rule Engine diganti total: 12 rule fisik dikonsolidasi lewat np.select
         menjadi 1 kolom `reason` + 1 kolom `severity_level`
         (NORMAL / WARNING / CRITICAL / FATAL-EMERGENCY).
      3) Definisi Thermal Lag diperketat: suhu harus TERUS turun tanpa rebound sama
         sekali selama jendela penuh 1 jam setelah heater menyala.
      4) XGBoost sekarang multi-class (4 kelas severity) — objective="multi:softprob".
      5) Seluruh UI (metric card, actionable alert, bar chart) mengonsumsi
         `reason` / `severity_level`, bukan RULE_COLS lama.
      6) Slider simulasi live sekarang punya tooltip fisik + catatan skenario
         Thermal Lag.
================================================================================
"""

import os
import json
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import joblib
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px

warnings.filterwarnings("ignore")

# ============================================================================
# 0. CONFIG
# ============================================================================
MODEL_PATH = "/kaggle/input/notebooks/aliciakyoumi/m15-cia-03-training"
DATA_PATH  = "/kaggle/input/notebooks/aliciakyoumi/m15-cia-02-processing/labeled_data.csv"

EXPECTED_N_FEATURES = 12
CORE_FEATURES = ["molten_temp", "heater1", "heater2", "voltage_avr", "current_avr", "power_total"]

# ---- [UPDATE #1] Physical thresholds (Al-Si phase diagram + electrical spec) ----
EUTECTIC_TEMP_C = 592        # di bawah ini: risiko logam membeku total (Eutectic)
LIQUIDUS_TEMP_C = 615        # 592-615 = Mushy Zone (fase transisi cair-padat)
VOLTAGE_NOMINAL_V = 380
VOLTAGE_MIN_V = 342          # -10% dari nominal
VOLTAGE_MAX_V = 418          # +10% dari nominal

# ---- [UPDATE #2] Severity levels (priority tinggi -> rendah) ----
SEV_FATAL, SEV_CRITICAL, SEV_WARNING, SEV_NORMAL = (
    "FATAL/EMERGENCY", "CRITICAL", "WARNING", "NORMAL",
)
SEVERITY_ORDER = [SEV_NORMAL, SEV_WARNING, SEV_CRITICAL, SEV_FATAL]
SEVERITY_TO_CLASS = {SEV_NORMAL: 0, SEV_WARNING: 1, SEV_CRITICAL: 2, SEV_FATAL: 3}
CLASS_TO_SEVERITY = {v: k for k, v in SEVERITY_TO_CLASS.items()}

# ---- [UPDATE #2] 12 physical rules -> reason strings (dipakai sbg key RULE_DETAILS) ----
R_EUTECTIC     = "Eutectic Limit Breach"
R_SHORT_CIRC   = "Short Circuit"
R_OPEN_CIRCUIT = "Heater Open Circuit / Broken Element"
R_STARVATION   = "Starvation Power (Thermal Lag + Voltage Drop)"
R_MUSHY_ZONE   = "Liquidus / Mushy Zone"
R_OVERHEAT     = "Overheat"
R_SENSOR_FAULT = "Sensor Fault - Molten Temp Unreadable"
R_ELEC_FAULT   = "Sensor Fault - Electrical Readings Unreadable"
R_THERMAL_LAG  = "Thermal Lag"
R_UNCONTROLLED = "Uncontrolled Heating"
R_LOW_TEMP     = "Low Temperature"
R_VOLTAGE_ABN  = "Voltage Abnormal"
R_NORMAL       = "Normal Operation"
R_ML_ONLY      = "ml_only"

# Urutan prioritas (paling parah dulu) dipakai oleh np.select & oleh bar chart.
# [Aligned to real Notebook 2 output] Short Circuit tetap FATAL/EMERGENCY; Overheat
# adalah CRITICAL (bukan WARNING); Sensor Fault adalah rule tersendiri (WARNING),
# dicek lebih dulu dari rule WARNING lain supaya baris dengan molten_temp tak
# terbaca tidak "ditutupi" oleh rule WARNING lain yang kebetulan ikut menyala.
# [Electrical fault handling] Heater Open Circuit (resistance ekstrem tinggi —
# current ~0 padahal voltage ada & heater terbaca ON, indikasi elemen putus) dan
# Sensor Fault - Electrical Readings (voltage_avr/current_avr/power_total serentak
# 0, indikasi meteran listrik tidak terbaca) ditambahkan berdasarkan pola nyata
# yang ditemukan pada data historis namun belum tertangani di pipeline sebelumnya.
REASON_ORDER = [R_EUTECTIC, R_SHORT_CIRC, R_OPEN_CIRCUIT, R_STARVATION, R_MUSHY_ZONE, R_OVERHEAT,
                R_SENSOR_FAULT, R_ELEC_FAULT, R_THERMAL_LAG, R_UNCONTROLLED, R_LOW_TEMP, R_VOLTAGE_ABN]

# ---- [UPDATE #4] Root-cause knowledge base (What / Where / Why / Recommendation) ----
# Dirangkum dari 'Ringkasan_Domain_Knowledge_Anomaly_Detection_HoldingFurnace.md'
# (Bagian B — Peta Indikator Anomali & Root Cause Analysis), diperluas dengan rule baru.
RULE_DETAILS = {
    R_EUTECTIC: {
        "what": "Eutectic Limit Breach — molten_temp_valid turun di bawah 592°C, mendekati "
                "titik beku total paduan Al-Si (eutectic point).",
        "where": "Sensor molten_temp (TC100 / Metal Temperature Controller).",
        "why": "Kegagalan pemanasan total (kedua heater mati/putus), heat-loss ekstrem "
               "(cover furnace terbuka lama), atau charging logam dingin dalam jumlah besar.",
        "recommendation": "EMERGENCY STOP proses tuang, nyalakan kedua heater ke daya maksimum, "
                           "dan periksa kondisi fisik heater serta cover furnace segera.",
    },
    R_SHORT_CIRC: {
        "what": "Short Circuit — resistance sangat rendah disertai lonjakan arus (Heating Element Failure).",
        "where": "Immersion Heater #1/#2 (protection tube) & Power Meter (current_avr, power_total).",
        "why": "Tube heater pecah dan molten metal masuk ke elemen (memicu Melt Leak Detect), isolasi "
               "elemen heater rusak/aus, atau SCR mengalami short pada sisi output.",
        "recommendation": "Lakukan EMERGENCY STOP, cek alarm 'Melt Leak Detect' & 'SCR Abnormal', dan "
                           "isolasi kelistrikan sebelum inspeksi fisik tube heater.",
    },
    R_OPEN_CIRCUIT: {
        "what": "Heater Open Circuit / Broken Element — current_avr mendekati 0 A padahal voltage_avr "
                "tetap normal dan heater terbaca ON, sesuai Hukum Ohm resistance melonjak ke Megaohm "
                "(rangkaian terputus).",
        "where": "Elemen Immersion Heater #1/#2 dan sambungan kabel/terminal menuju heater.",
        "why": "Elemen heater putus total (kawat pemanas terbakar/patah), sambungan kabel lepas/korosi, "
               "atau fuse/breaker pada jalur heater tersebut trip tanpa mematikan status kontrol.",
        "recommendation": "Cek kontinuitas elemen heater dengan multimeter, periksa sambungan terminal & "
                           "fuse/breaker, dan jadwalkan penggantian elemen bila terkonfirmasi putus.",
    },
    R_STARVATION: {
        "what": "Starvation Power — Thermal Lag terjadi BERSAMAAN dengan voltage_avr di luar rentang "
                "342V-418V (indikasi suplai daya ke heater tidak mencukupi).",
        "where": "Panel SCR/Thyristor Heater #1/#2 & jalur suplai voltage_avr.",
        "why": "Drop tegangan pada jalur suplai (beban listrik pabrik tinggi / gangguan trafo) "
               "membuat heater tidak mampu mengeluarkan daya penuh meski secara status ON.",
        "recommendation": "Cek panel distribusi listrik & beban pada jalur yang sama, laporkan ke tim "
                           "kelistrikan pabrik, dan pantau voltage_avr secara kontinu.",
    },
    R_MUSHY_ZONE: {
        "what": "Liquidus / Mushy Zone — molten_temp_valid berada di 592-615°C, fase transisi "
                "cair-padat (semi-solid) pada diagram fasa Al-Si.",
        "where": "Metal Temperature Controller (TC100 / TIC100).",
        "why": "Heat-loss melebihi kapasitas heater, thermal lag berkepanjangan, atau setpoint "
               "holding temperature terlalu rendah untuk kondisi operasi saat ini.",
        "recommendation": "Naikkan daya heater segera, tunda proses tuang sampai suhu kembali di atas "
                           "615°C, dan cek isolasi/cover furnace.",
    },
    R_THERMAL_LAG: {
        "what": "Thermal Lag / Heating Failure — heater menyala namun molten_temp_valid terus turun "
                "TANPA rebound sama sekali selama jendela penuh 1 jam.",
        "where": "Elemen Immersion Heater #1/#2 dan lapisan refractory furnace.",
        "why": "Elemen heater aus (masih terbaca ON secara switch tapi tidak menghantarkan panas), tube "
               "retak/leakage, kapasitas 16 kW tidak mencukupi laju heat-loss, atau SCR partial failure.",
        "recommendation": "Jadwalkan preventive maintenance heater & refractory, pastikan cover furnace "
                           "tertutup rapat, dan pantau tren suhu 30-60 menit ke depan.",
    },
    R_UNCONTROLLED: {
        "what": "Uncontrolled Heating — molten_temp naik meski heater #1 dan #2 dalam kondisi OFF.",
        "where": "Sensor molten_temp (TC100 / Metal Temperature Controller) & status kontak SCR Heater #1/#2.",
        "why": "Indikasi SCR/Thyristor gagal 'fail-on' (tetap mengalirkan daya walau perintah OFF), "
               "thermocouple drift, atau sumber panas eksternal yang tidak terkontrol.",
        "recommendation": "Segera periksa kondisi SCR 200/300 Heater #1 & #2, verifikasi pembacaan "
                           "thermocouple TC100 secara manual, dan jadwalkan inspeksi kelistrikan.",
    },
    R_OVERHEAT: {
        "what": "Metal Temperature Overheat — molten_temp_valid melampaui ambang batas atas.",
        "where": "Metal Temperature Controller (TC100 / TIC100).",
        "why": "Thermocouple bermasalah, atau setpoint holding temperature tidak sesuai kebutuhan proses.",
        "recommendation": "Verifikasi setpoint controller & kalibrasi thermocouple, turunkan daya heater "
                           "sementara sampai suhu kembali ke rentang normal (640-660°C).",
    },
    R_LOW_TEMP: {
        "what": "Metal Temperature Low — molten_temp_valid berada di bawah ambang batas bawah "
                "(namun masih di atas Liquidus 615°C).",
        "where": "Metal Temperature Controller (TC100 / TIC100).",
        "why": "Thermocouple bermasalah, atau kegagalan pemanasan (berkaitan erat dengan Thermal Lag).",
        "recommendation": "Cek status kedua heater & SCR, pastikan tidak ada charging molten metal dingin "
                           "berlebih, dan tingkatkan frekuensi monitoring suhu tuang.",
    },
    R_VOLTAGE_ABN: {
        "what": "Voltage Abnormal — voltage_avr di luar rentang normal 342V-418V (nominal 380V ±10%) "
                "tanpa disertai Thermal Lag.",
        "where": "Panel suplai listrik utama & AVR (Automatic Voltage Regulator) furnace.",
        "why": "Fluktuasi tegangan jaringan pabrik, gangguan AVR, atau beban listrik berlebih pada "
               "jalur yang sama.",
        "recommendation": "Pantau tren voltage_avr, laporkan ke tim kelistrikan bila fluktuasi berulang, "
                           "dan waspadai potensi Starvation Power bila suhu mulai ikut turun.",
    },
    R_SENSOR_FAULT: {
        "what": "Sensor Fault - Molten Temp Unreadable — molten_temp terbaca 0 (secara fisik mustahil "
                "selama operasi), menandakan thermocouple TC100 gagal mengirim pembacaan valid.",
        "where": "Thermocouple / Metal Temperature Controller (TC100 / TIC100) dan wiring sensor terkait.",
        "why": "Thermocouple lepas/putus, koneksi wiring longgar, gangguan pada modul input controller, "
               "atau gangguan komunikasi data logger.",
        "recommendation": "Verifikasi pembacaan suhu secara manual di panel mesin, periksa koneksi "
                           "thermocouple, dan JANGAN mengandalkan rule berbasis suhu (Overheat/Low Temp/"
                           "Thermal Lag) selama pembacaan ini belum pulih.",
    },
    R_ELEC_FAULT: {
        "what": "Sensor Fault - Electrical Readings Unreadable — voltage_avr, current_avr, DAN power_total "
                "serentak terbaca 0, padahal voltage suplai seharusnya tetap ~380V meski heater OFF.",
        "where": "Power meter / AVR pada panel distribusi listrik furnace.",
        "why": "Modul power meter kehilangan komunikasi data (data tidak terbaca), CT (current "
               "transformer)/PT (potential transformer) lepas, atau gangguan pada data logger listrik.",
        "recommendation": "Verifikasi pembacaan panel listrik secara manual, cek koneksi power meter & "
                           "CT/PT, dan JANGAN mengandalkan rule berbasis current/power/resistance selama "
                           "pembacaan ini belum pulih.",
    },
    # Fallback ketika Layer-1 rule TIDAK menyala tapi ensemble ML tetap mendeteksi anomali
    R_ML_ONLY: {
        "what": "Pola Tidak Normal Terdeteksi oleh Model ML (Rule Layer-1 belum menyala).",
        "where": "Kombinasi parameter molten_temp, current_avr, power_total, dan resistance secara "
                 "keseluruhan (bukan satu sensor tunggal).",
        "why": "OCSVM/LOF/XGBoost mendeteksi kombinasi nilai yang menyimpang dari pola operasi normal "
               "historis, meski belum melewati ambang batas rule deterministik.",
        "recommendation": "Pantau tren 15-30 menit ke depan, lakukan konfirmasi manual pada panel mesin, "
                           "dan siapkan inspeksi bila skor anomali terus meningkat.",
    },
}

# ---- Palet warna tema PINK (nilai default — akan ditimpa oleh pemilihan
#      tema di sidebar sebelum inject_css() dipanggil, lihat bagian 6) --------
C_PRIMARY, C_SECONDARY, C_LIGHT = "#EC4899", "#F472B6", "#FBCFE8"
C_LILAC_BG, C_CARD, C_INK, C_MUTED = "#FFF0F6", "#FFFFFF", "#6B1D42", "#B8749A"
C_BORDER, C_GRID, C_SIDEBAR_BG = "#FBDCEA", "#FCE7F1", "#FFFAFC"
C_DANGER, C_WARNING, C_SUCCESS = "#E53E3E", "#DD6B20", "#38A169"
C_HEADER_TEXT, C_IS_DARK = "#FFFFFF", False

st.set_page_config(
    page_title="Holding Furnace — Predictive Maintenance",
    page_icon="🌸",
    layout="wide",
    initial_sidebar_state="expanded",
)


# ============================================================================
# 0a. TIMEZONE — Asia/Jakarta (WIB), real-time
# ============================================================================
try:
    from zoneinfo import ZoneInfo  # stdlib sejak Python 3.9 (tersedia di Kaggle)
    JAKARTA_TZ = ZoneInfo("Asia/Jakarta")
except Exception:
    import pytz  # fallback untuk environment Python lama tanpa zoneinfo
    JAKARTA_TZ = pytz.timezone("Asia/Jakarta")


def now_wib() -> datetime:
    """Waktu saat ini di zona Asia/Jakarta (WIB), timezone-aware."""
    return datetime.now(JAKARTA_TZ)


# ============================================================================
# 0b. THEME ENGINE — Light / Dark / Custom dengan kontras teks otomatis
# ============================================================================
def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def _rgb_to_hex(rgb):
    return "#%02X%02X%02X" % tuple(max(0, min(255, int(round(c)))) for c in rgb)


def mix_hex(c1, c2, t):
    """Campur dua warna hex; t=0 -> c1 penuh, t=1 -> c2 penuh."""
    a, b = _hex_to_rgb(c1), _hex_to_rgb(c2)
    return _rgb_to_hex(tuple(a[i] + (b[i] - a[i]) * t for i in range(3)))


def relative_luminance(hexcolor):
    r, g, b = (c / 255 for c in _hex_to_rgb(hexcolor))
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def contrast_text(bg_hex, dark_txt="#1A1523", light_txt="#FFFFFF"):
    """Pilih teks hitam/putih yang paling kontras terhadap warna bg_hex."""
    return light_txt if relative_luminance(bg_hex) < 0.55 else dark_txt


def build_theme(primary, secondary, light, bg, card, ink):
    """Derive semua warna turunan (muted/border/grid/sidebar/status) dari 6
    warna dasar, supaya kontras tulisan selalu terjaga baik untuk tema
    Light, Dark, maupun Custom (warna apa pun yang dipilih pengguna)."""
    is_dark = relative_luminance(bg) < 0.5
    muted = mix_hex(ink, bg, 0.45)
    border = mix_hex(ink, bg, 0.88)
    grid = mix_hex(ink, bg, 0.92)
    sidebar_bg = mix_hex(card, bg, 0.35)
    danger, warning, success = (("#FC8181", "#F6AD55", "#68D391") if is_dark
                                 else ("#E53E3E", "#DD6B20", "#38A169"))
    header_text = contrast_text(mix_hex(primary, secondary, 0.5))
    return dict(primary=primary, secondary=secondary, light=light, bg=bg, card=card, ink=ink,
                muted=muted, border=border, grid=grid, sidebar_bg=sidebar_bg,
                danger=danger, warning=warning, success=success,
                header_text=header_text, is_dark=is_dark)


def build_pink_scale(bg, primary, steps=5):
    """Membangun color scale pink yang MONOTON secara luminance, dari bg
    tema aktif menuju primary yang di-darken 35% ke hitam."""
    primary_dark = mix_hex(primary, "#000000", 0.35)
    stops = [mix_hex(bg, primary_dark, t) for t in np.linspace(0, 1, steps)]
    return sorted(stops, key=relative_luminance, reverse=True)


# Definisi awal PINK_SCALE (default, sebelum tema dipilih user).
PINK_SCALE = build_pink_scale(C_LILAC_BG, C_PRIMARY)

# Warna badge per severity level (dihitung ulang setiap kali tema berubah, lihat bag. 8)
SEVERITY_COLOR_MAP = {}

PRESET_BASE = {
    "🌸 Light": dict(primary="#EC4899", secondary="#F472B6", light="#FBCFE8",
                              bg="#FFF0F6", card="#FFFFFF", ink="#6B1D42"),
    "🌙 Dark":   dict(primary="#F9A8D4", secondary="#F472B6", light="#FCE7F3",
                              bg="#241019", card="#361628", ink="#FDE8F1"),
}

# ============================================================================
# 1. CUSTOM CSS
# ============================================================================
def inject_css():
    # Warna turunan (tint) dihitung dari bg halaman saat ini, supaya semua
    # kartu/badge/box tetap kontras & konsisten baik di tema Light, Dark,
    # maupun Custom (warna apa pun yang dipilih pengguna).
    badge_danger_bg = mix_hex(C_DANGER, C_LILAC_BG, 0.85)
    badge_ok_bg     = mix_hex(C_SUCCESS, C_LILAC_BG, 0.85)
    badge_off_bg    = mix_hex(C_MUTED, C_LILAC_BG, 0.85)
    badge_warn_bg   = mix_hex(C_WARNING, C_LILAC_BG, 0.85)
    alert_bg        = mix_hex(C_DANGER, C_LILAC_BG, 0.88)
    off_bg          = mix_hex(C_MUTED, C_LILAC_BG, 0.88)
    rule_off_bg     = mix_hex(C_MUTED, C_LILAC_BG, 0.85)
    rule_on_text    = contrast_text(C_DANGER)
    rec_accent      = "#F6E05E" if C_IS_DARK else "#ECC94B"
    rec_bg          = mix_hex(rec_accent, C_LILAC_BG, 0.85)
    rec_text        = "#FEFCBF" if C_IS_DARK else "#6B5900"
    shadow_col      = "rgba(0,0,0,0.35)" if C_IS_DARK else "rgba(107,70,193,0.10)"

    st.markdown(f"""
    <style>
        :root {{
            --app-primary: {C_PRIMARY};
            --app-ink: {C_INK};
            --app-muted: {C_MUTED};
        }}
        .stApp {{ background-color: {C_LILAC_BG}; }}
        #MainMenu, footer {{visibility: hidden;}}

        /* -- teks native Streamlit ikut menyesuaikan tema (custom + fallback var) -- */
        .stApp, .stApp p, .stApp li, .stApp label, [data-testid="stMarkdownContainer"] p {{
            color: {C_INK};
        }}
        [data-testid="stCaptionContainer"] {{ color: {C_MUTED} !important; }}
        [data-testid="stWidgetLabel"] p {{ color: {C_INK}; font-weight: 600; }}
        [data-testid="stTabs"] button p {{ color: {C_MUTED}; font-weight: 600; }}
        [data-testid="stTabs"] button[aria-selected="true"] p {{ color: {C_PRIMARY}; }}
        [data-testid="stDataFrame"] {{ color: var(--text-color, {C_INK}); }}
        [data-testid="stFileUploader"] section {{ background: {C_CARD}; border-color: {C_BORDER}; }}
        hr {{ border-color: {C_BORDER}; }}

        .app-header {{
            background: linear-gradient(90deg, {C_PRIMARY} 0%, {C_SECONDARY} 100%);
            padding: 22px 28px; border-radius: 14px; margin-bottom: 18px;
            box-shadow: 0 6px 18px rgba(0,0,0,0.20);
        }}
        .app-header h1 {{ color: {C_HEADER_TEXT}; margin: 0; font-size: 26px; font-weight: 800; }}
        .app-header p  {{ color: {C_HEADER_TEXT}; opacity: 0.85; margin: 2px 0 0 0; font-size: 13px; }}

        .metric-card {{
            background: {C_CARD}; border-radius: 14px; padding: 16px 18px;
            border-left: 6px solid {C_PRIMARY};
            box-shadow: 0 3px 10px {shadow_col};
            margin-bottom: 10px;
        }}
        .metric-card.warn  {{ border-left-color: {C_WARNING}; }}
        .metric-card.danger{{ border-left-color: {C_DANGER}; }}
        .metric-card.ok    {{ border-left-color: {C_SUCCESS}; }}
        .metric-label {{ color: {C_MUTED}; font-size: 12px; font-weight: 600; text-transform: uppercase; letter-spacing:.4px;}}
        .metric-value {{ color: {C_INK}; font-size: 28px; font-weight: 800; margin: 2px 0 0 0; }}
        .metric-sub   {{ font-size: 12px; font-weight: 600; margin-top: 4px; }}

        .card {{
            background: {C_CARD}; border-radius: 14px; padding: 18px 20px;
            box-shadow: 0 3px 10px {shadow_col}; margin-bottom: 14px;
        }}
        .card h4 {{ color: {C_PRIMARY}; margin: 0 0 10px 0; font-size: 14px;
                    text-transform: uppercase; letter-spacing: .5px; font-weight: 800;}}

        .badge {{ padding: 5px 14px; border-radius: 999px; font-size: 12px; font-weight: 700; display:inline-block;}}
        .badge-danger  {{ background:{badge_danger_bg}; color:{C_DANGER}; }}
        .badge-ok      {{ background:{badge_ok_bg}; color:{C_SUCCESS}; }}
        .badge-off     {{ background:{badge_off_bg}; color:{C_MUTED}; }}
        .badge-warn    {{ background:{badge_warn_bg}; color:{C_WARNING}; }}

        .rec-box {{
            background:{rec_bg}; border-left: 5px solid {rec_accent}; border-radius:10px;
            padding: 12px 16px; font-size: 13px; color:{rec_text};
        }}
        .alert-box {{
            background:{alert_bg}; border-left:5px solid {C_DANGER}; border-radius:10px;
            padding:12px 16px; font-size:13px; color:{C_DANGER}; margin-bottom:14px;
        }}
        .off-box {{
            background:{off_bg}; border-left:5px solid {C_MUTED}; border-radius:10px;
            padding:12px 16px; font-size:13px; color:{C_MUTED}; margin-bottom:14px;
        }}
        .rule-chip {{
            display:inline-block; padding:4px 10px; border-radius:8px; font-size:12px;
            font-weight:700; margin:2px 4px 2px 0;
        }}
        .rule-on  {{ background:{C_DANGER}; color:{rule_on_text}; }}
        .rule-off {{ background:{rule_off_bg}; color:{C_MUTED}; }}

        /* Badge severity_level (NORMAL/WARNING/CRITICAL/FATAL) — warna diinject dinamis
           lewat inline-style pada pemanggilan (lihat severity_badge()). */
        .severity-badge {{
            padding: 5px 16px; border-radius: 999px; font-size: 13px; font-weight: 800;
            display:inline-block; letter-spacing:.3px;
        }}

        /* [UPDATE #4] Alert box "actionable" 4-bagian: What / Where / Why / Recommendation */
        .actionable-box {{
            background:{C_CARD}; border:1px solid {C_BORDER}; border-left:6px solid {C_DANGER};
            border-radius:12px; padding:16px 18px; margin-bottom:14px;
            box-shadow: 0 3px 10px {shadow_col};
        }}
        .actionable-box .aa-title {{ color:{C_DANGER}; font-weight:800; font-size:14px; margin-bottom:8px; }}
        .actionable-box .aa-row {{ margin-bottom:6px; font-size:13px; color:{C_INK}; }}
        .actionable-box .aa-row b {{ color:{C_PRIMARY}; }}

        section[data-testid="stSidebar"] {{ background-color: {C_SIDEBAR_BG}; border-right: 1px solid {C_BORDER};}}
        section[data-testid="stSidebar"] h3 {{ color:{C_PRIMARY}; font-size:12px; text-transform:uppercase;
                                                 letter-spacing:.5px; font-weight:800; margin-top:18px;}}
        div[data-testid="stMetricValue"] {{ color:{C_INK}; }}
        .stButton>button {{
            border-radius: 10px; font-weight: 700; border: 1px solid {C_BORDER};
            background: {C_CARD}; color: {C_INK};
        }}
        .stDownloadButton>button {{
            border-radius: 10px; font-weight: 700; border: 1px solid {C_PRIMARY};
            background: {C_PRIMARY}; color: {contrast_text(C_PRIMARY)};
        }}
    </style>
    """, unsafe_allow_html=True)


def metric_card(label, value, unit="", sub=None, status="normal"):
    css_class = {"normal": "", "warn": "warn", "danger": "danger", "ok": "ok"}.get(status, "")
    sub_color = {"normal": C_MUTED, "warn": C_WARNING, "danger": C_DANGER, "ok": C_SUCCESS}.get(status, C_MUTED)
    sub_html = f'<div class="metric-sub" style="color:{sub_color}">{sub}</div>' if sub else ""
    st.markdown(f"""
        <div class="metric-card {css_class}">
            <div class="metric-label">{label}</div>
            <div class="metric-value">{value}<span style="font-size:14px; font-weight:600; color:{C_MUTED};"> {unit}</span></div>
            {sub_html}
        </div>
    """, unsafe_allow_html=True)


def progress_row(label, value, max_value, unit="", color=None):
    color = color or C_PRIMARY  # resolved at call-time so it follows the active theme
    pct = 0 if max_value == 0 else max(0, min(100, (value / max_value) * 100))
    st.markdown(f"""
        <div style="margin-bottom:10px;">
            <div style="display:flex; justify-content:space-between; font-size:12px; color:{C_INK};">
                <span style="color:{C_MUTED}; font-weight:600;">{label}</span>
                <span style="font-weight:800;">{value:,.2f} {unit}</span>
            </div>
            <div style="background:{C_GRID}; border-radius:8px; height:8px; margin-top:4px;">
                <div style="width:{pct}%; background:{color}; height:8px; border-radius:8px;"></div>
            </div>
        </div>
    """, unsafe_allow_html=True)


def severity_badge(severity_level):
    """[UPDATE #2] Badge berwarna untuk severity_level (NORMAL/WARNING/CRITICAL/FATAL)."""
    bg = SEVERITY_COLOR_MAP.get(severity_level, C_MUTED)
    icon = {SEV_NORMAL: "✅", SEV_WARNING: "⚠️", SEV_CRITICAL: "🟠", SEV_FATAL: "🚨"}.get(severity_level, "•")
    st.markdown(
        f'<span class="severity-badge" style="background:{mix_hex(bg, C_LILAC_BG, 0.80)}; color:{bg};">'
        f'{icon} {severity_level}</span>',
        unsafe_allow_html=True,
    )


def actionable_alert(reason_list, extra_note=""):
    """[UPDATE #4] Render 1..n kartu 'What/Where/Why/Recommendation' untuk tiap
    reason yang aktif. Jika tidak ada rule yang menyala (murni ML), gunakan
    entri fallback 'ml_only'."""
    keys = [r for r in reason_list if r and r != R_NORMAL] or [R_ML_ONLY]
    seen = set()
    for k in keys:
        if k in seen:
            continue
        seen.add(k)
        d = RULE_DETAILS.get(k, RULE_DETAILS[R_ML_ONLY])
        st.markdown(f"""
            <div class="actionable-box">
                <div class="aa-title">🚨 {d['what'].split(' — ')[0]}</div>
                <div class="aa-row"><b>What:</b> {d['what']}</div>
                <div class="aa-row"><b>Where:</b> {d['where']}</div>
                <div class="aa-row"><b>Why:</b> {d['why']}</div>
                <div class="aa-row"><b>Recommendation:</b> {d['recommendation']}</div>
            </div>
        """, unsafe_allow_html=True)
    if extra_note:
        st.caption(extra_note)


# ============================================================================
# 2. LOAD MODEL ARTIFACTS  (cached — loaded once per session)
# ============================================================================
@st.cache_resource(show_spinner="Memuat model AI (OCSVM, LOF, XGBoost)...")
def load_artifacts(model_path):
    ocsvm  = joblib.load(os.path.join(model_path, "model_ocsvm.pkl"))
    lof    = joblib.load(os.path.join(model_path, "model_lof.pkl"))
    clf    = joblib.load(os.path.join(model_path, "model_xgb.pkl"))
    scaler = joblib.load(os.path.join(model_path, "scaler.pkl"))
    with open(os.path.join(model_path, "model_metadata.json")) as f:
        metadata = json.load(f)
    return ocsvm, lof, clf, scaler, metadata


@st.cache_data(show_spinner="Memuat data historis mesin...")
def load_historical_data(data_path):
    df = pd.read_csv(data_path, parse_dates=["timestamp"])
    return df


# ---- Mode Demo: bangkitkan data + model sintetis jika artifact asli hilang ----
@st.cache_resource(show_spinner="Menyiapkan Mode Demo (data & model sintetis)...")
def build_demo_artifacts(n=4000, seed=42):
    """Membuat dataset & model sintetis yang mereplikasi persis pipeline
    Notebook 2 (feature engineering + rule labeling) & Notebook 3 (training),
    supaya dashboard tetap bisa langsung dijalankan tanpa artifact Kaggle asli."""
    rng = np.random.default_rng(seed)
    ts = pd.date_range("2025-06-01", periods=n, freq="1min")

    molten_temp = 650 + 4 * np.sin(np.linspace(0, 40, n)) + rng.normal(0, 1.5, n)
    heater1 = np.clip(rng.normal(50, 20, n), 0, 100)
    heater2 = np.clip(rng.normal(50, 20, n), 0, 100)
    voltage_avr = 380 + rng.normal(0, 4, n)
    current_avr = np.clip(120 + 10 * np.sin(np.linspace(0, 20, n)) + rng.normal(0, 5, n), 0, None)
    power_total = np.clip(45 + 5 * np.sin(np.linspace(0, 15, n)) + rng.normal(0, 3, n), 0, None)

    # Suntikkan beberapa episode anomali sintetis (mencakup tiap kelompok severity)
    for start in rng.choice(np.arange(200, n - 200), size=14, replace=False):
        kind = rng.integers(0, 6)
        span = slice(start, start + rng.integers(15, 40))
        if kind == 0:  # overheat (WARNING)
            molten_temp[span] += rng.uniform(25, 45)
        elif kind == 1:  # low temp (WARNING, tetap di atas liquidus)
            molten_temp[span] -= rng.uniform(15, 24)
        elif kind == 2:  # short circuit signature (FATAL)
            current_avr[span] += rng.uniform(150, 250)
            power_total[span] = power_total[start]
        elif kind == 3:  # uncontrolled heating (WARNING)
            heater1[span] = 0
            heater2[span] = 0
            molten_temp[span] += np.linspace(0, 20, span.stop - span.start)
        elif kind == 4:  # mushy zone / eutectic dive (CRITICAL/FATAL) — heater tetap ON
            molten_temp[span] -= np.linspace(0, 65, span.stop - span.start)
        else:  # voltage drop / starvation power (CRITICAL/WARNING)
            voltage_avr[span] -= rng.uniform(50, 70)
            molten_temp[span] -= np.linspace(0, 20, span.stop - span.start)

    # Sensor fault sintetis: beberapa titik molten_temp == 0 (harus jadi molten_temp_valid=NaN)
    fault_idx = rng.choice(np.arange(n), size=15, replace=False)
    molten_temp[fault_idx] = 0.0

    df = pd.DataFrame({
        "timestamp": ts, "molten_temp": molten_temp, "heater1": heater1, "heater2": heater2,
        "voltage_avr": voltage_avr, "current_avr": current_avr, "power_total": power_total,
    })

    df = engineer_batch_features(df, samples_per_hour=60)
    metadata_seed = {"overheat_threshold": 680, "low_temp_threshold": 640, "thermal_lag_delta": -5}
    df = apply_layer1_rules_batch(df, metadata_seed, samples_per_hour=60)

    FEATURE_COLS = ["molten_temp", "heater1", "heater2", "voltage_avr", "current_avr", "power_total",
                     "resistance", "delta_temp_1h", "heater_active", "heater_both_off",
                     "current_avr_roll_std", "power_total_roll_std"]

    resistance_sentinel = df["resistance"].max() * 2 if df["resistance"].notna().any() else 2.0
    df_model = df.copy()
    df_model["resistance"] = df_model["resistance"].fillna(resistance_sentinel)
    df_model = df_model.dropna(subset=FEATURE_COLS + ["is_anomaly", "severity_level"]).reset_index(drop=True)

    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import OneClassSVM
    from sklearn.neighbors import LocalOutlierFactor

    X = df_model[FEATURE_COLS]
    y_binary = df_model["is_anomaly"]
    # [UPDATE #4] Target multi-class XGBoost (0=NORMAL, 1=WARNING, 2=CRITICAL, 3=FATAL/EMERGENCY)
    y_multiclass = df_model["severity_level"].map(SEVERITY_TO_CLASS).astype(int)

    scaler = StandardScaler().fit(X)
    X_scaled = scaler.transform(X)

    ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit(X_scaled[y_binary.values == 0])
    lof = LocalOutlierFactor(n_neighbors=25, contamination=0.05, novelty=True).fit(X_scaled)

    try:
        from xgboost import XGBClassifier
        clf = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.08,
                             subsample=0.8, colsample_bytree=0.8,
                             objective="multi:softprob", num_class=4,
                             eval_metric="mlogloss", random_state=42)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        clf = HistGradientBoostingClassifier(max_depth=5, random_state=42)
    clf.fit(X, y_multiclass)

    metadata = {
        "feature_cols": FEATURE_COLS,
        "core_features": CORE_FEATURES,
        "overheat_threshold": 680,
        "low_temp_threshold": 640,
        "thermal_lag_delta": -5,
        "resistance_sentinel": float(resistance_sentinel),
        "resistance_low_q10": float(df_model["resistance"].quantile(0.10)),
        "resistance_high_q90": float(df_model["resistance"].quantile(0.90)),
        "power_roll_std_q20": float(df_model["power_total_roll_std"].quantile(0.20)),
        "xgb_num_class": 4,
        "demo_mode": True,
    }
    return ocsvm, lof, clf, scaler, metadata, df_model


# ============================================================================
# 3. CLEANSING PIPELINE — mereplikasi Notebook 1 (m15-cia-01-cleansing)
# ============================================================================
def cleanse_raw_data(df_raw, core_features=None, start_date="2025-03-01", end_date="2025-09-01"):
    """
    Mereplikasi aturan domain cleansing pada Notebook 1, dijalankan otomatis
    di backend saat pengguna meng-upload data mentah (mode 'Upload File'):
      1. Deteksi & parse kolom timestamp, urutkan secara kronologis.
      2. Time slicing -> hanya periode Maret-Agustus 2025 yang dipertahankan
         (periode di luar rentang ini otomatis DILEWATI jika file upload
         berada sepenuhnya di luar rentang tsb, supaya data tidak hilang total).
      3. RULE Full-Zero -> drop baris jika SELURUH core_features == 0 sekaligus
         (indikasi mesin/sensor mati total).
      4. RULE Partial-Zero -> TIDAK di-drop (baris ini justru kunci identifikasi
         anomali, mis. power_total=0 tapi molten_temp masih terbaca; molten_temp=0
         sendiri ditangani terpisah sebagai sensor fault di engineer_*_features()).
      5. Forward-fill gap pendek (limit=3 baris) untuk dropout sensor sesaat;
         gap panjang dibiarkan NaN (tidak difabrikasi).
    Mengembalikan (df_clean, cleaning_log) untuk ditampilkan sebagai audit trail.
    """
    core_features = core_features or CORE_FEATURES
    df = df_raw.copy()

    time_col_candidates = ["timestamp", "datetime", "date", "time", "Timestamp", "Datetime"]
    time_col = next((c for c in time_col_candidates if c in df.columns), df.columns[0])
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
    df = df.rename(columns={time_col: "timestamp"})

    missing_cols = [c for c in core_features if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Kolom wajib tidak ditemukan pada file upload: {missing_cols}")

    n_raw = len(df)

    # 2. Time slicing (dilewati bila TIDAK ADA baris sama sekali dalam rentang,
    #    supaya data live/terbaru di luar Mar-Aug 2025 tidak terhapus habis)
    in_range_mask = (df["timestamp"] >= start_date) & (df["timestamp"] < end_date)
    if in_range_mask.any():
        df = df[in_range_mask].reset_index(drop=True)
        time_slice_applied = True
    else:
        time_slice_applied = False
    n_after_slice = len(df)

    # 3. Full-Zero Rule
    full_zero_mask = (df[core_features] == 0).all(axis=1)
    n_full_zero = int(full_zero_mask.sum())
    df = df[~full_zero_mask].reset_index(drop=True)
    n_after_zero = len(df)

    # 4. Partial-zero rows (termasuk molten_temp==0 sensor fault) sengaja TIDAK
    #    disentuh di sini — ditangani di engineer_*_features() lewat molten_temp_valid.

    # 5. Forward-fill gap pendek
    df[core_features] = df[core_features].ffill(limit=3)
    remaining_na = int(df[core_features].isna().sum().sum())

    cleaning_log = {
        "rows_raw": n_raw,
        "time_slice_applied": time_slice_applied,
        "rows_after_time_slice": n_after_slice,
        "rows_dropped_full_zero": n_full_zero,
        "rows_final": n_after_zero,
        "remaining_nan_after_ffill": remaining_na,
    }
    return df, cleaning_log


# ============================================================================
# 4. FEATURE ENGINEERING — batch version (mirrors Notebook 2, for demo/history/upload)
# ============================================================================
def engineer_batch_features(df, samples_per_hour=60):
    df = df.sort_values("timestamp").reset_index(drop=True)

    # [UPDATE #1] Sensor Fault Handling — molten_temp == 0 secara fisik mustahil
    # selama operasi normal (bukan temperature drop asli), jadi ditandai sebagai
    # sensor fault dan di-NaN-kan di kolom molten_temp_valid. SEMUA fitur turunan
    # (delta_temp_1h, rule threshold, dst) WAJIB memakai molten_temp_valid,
    # bukan molten_temp mentah, supaya tidak memicu false "extreme drop".
    df["molten_temp_sensor_fault_flag"] = (df["molten_temp"] == 0).astype(int)
    df["molten_temp_valid"] = df["molten_temp"].replace(0, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        df["resistance"] = np.where(df["current_avr"] == 0, np.nan,
                                     df["power_total"] / (df["current_avr"] ** 2))
    df["delta_temp_1h"] = (df["molten_temp_valid"] - df["molten_temp_valid"].shift(samples_per_hour)).fillna(0)
    df["heater_active"] = ((df["heater1"] > 0) | (df["heater2"] > 0)).astype(int)
    df["heater_both_off"] = ((df["heater1"] == 0) & (df["heater2"] == 0)).astype(int)
    df["current_avr_roll_std"] = df["current_avr"].rolling(samples_per_hour, min_periods=1).std().fillna(0)
    df["power_total_roll_std"] = df["power_total"].rolling(samples_per_hour, min_periods=1).std().fillna(0)
    roll_mean_c = df["current_avr"].rolling(samples_per_hour * 3, min_periods=1).mean()
    roll_std_c = df["current_avr"].rolling(samples_per_hour * 3, min_periods=1).std()
    df["current_avr_spike"] = (df["current_avr"] > (roll_mean_c + 2 * roll_std_c)).fillna(False)
    df["power_stagnant"] = df["power_total_roll_std"] < df["power_total_roll_std"].quantile(0.20)
    return df


def apply_layer1_rules_batch(df, metadata, samples_per_hour=4):
    """[UPDATE #2] Layer-1 Rule Engine — 12 rule fisik dikonsolidasi lewat
    np.select (priority-based) menjadi satu kolom `reason` + satu kolom
    `severity_level` (NORMAL / WARNING / CRITICAL / FATAL-EMERGENCY).
    is_anomaly = 1 jika severity_level != NORMAL."""
    overheat_th = metadata.get("overheat_threshold", 680)
    low_th = metadata.get("low_temp_threshold", 640)
    # Magnitude floor for Thermal Lag — required because raw molten_temp is a noisy
    # ±1-3°C signal with the heater essentially always ON, so "any" non-increasing
    # 1-hour run happens by chance constantly (verified against real Notebook 2
    # output: a bare `< 0` floor false-fires on ~1/3 of all rows vs. 0 real Thermal
    # Lag events). A meaningful net drop is required before it counts as lag.
    lag_delta = metadata.get("thermal_lag_delta", -5)

    mt = df["molten_temp_valid"]

    # ---- [UPDATE #3] Thermal Lag — suhu TERUS turun tanpa rebound sama sekali
    # selama jendela penuh 1 jam (samples_per_hour sampel) setelah heater ON, DAN
    # net drop selama jendela tsb melewati lag_delta (default -5°C).
    diff1 = mt.diff()
    no_rebound_step = (diff1 <= 0).fillna(False)
    no_rebound_full_window = (no_rebound_step.rolling(samples_per_hour, min_periods=samples_per_hour)
                               .min().fillna(0).astype(bool))
    heater_active_full_window = (df["heater_active"].rolling(samples_per_hour, min_periods=samples_per_hour)
                                  .min().fillna(0).astype(bool))
    delta_full_window = mt - mt.shift(samples_per_hour)
    thermal_lag_base = (no_rebound_full_window & heater_active_full_window &
                         (delta_full_window <= lag_delta).fillna(False))

    # ---- Voltage abnormal (nominal 380V, toleransi 342-418V) ----
    voltage_abnormal = ((df["voltage_avr"] < VOLTAGE_MIN_V) | (df["voltage_avr"] > VOLTAGE_MAX_V)).fillna(False)

    starvation_power_cond = thermal_lag_base & voltage_abnormal
    thermal_lag_only_cond = thermal_lag_base & ~voltage_abnormal
    voltage_abnormal_only_cond = voltage_abnormal & ~thermal_lag_base

    # ---- Eutectic / Mushy Zone (diagram fasa Al-Si) ----
    eutectic_cond = (mt < EUTECTIC_TEMP_C).fillna(False)
    mushy_zone_cond = ((mt >= EUTECTIC_TEMP_C) & (mt < LIQUIDUS_TEMP_C)).fillna(False)

    # ---- Short Circuit / Heating Element Failure (resistance sangat rendah) ----
    valid_r = df["resistance"].dropna()
    low_thresh_relaxed = valid_r.quantile(0.10) if len(valid_r) else 0
    high_thresh_relaxed = valid_r.quantile(0.90) if len(valid_r) else np.inf
    short_circuit_cond = ((df["resistance"] < low_thresh_relaxed) & df["power_stagnant"] &
                           df["current_avr_spike"]).fillna(False)

    # ---- [Electrical fault handling] resistance_class untuk keperluan display/debug ----
    # (Very Low = risiko Short Circuit; Very High = risiko Open Circuit / heater putus;
    # Unknown = current_avr == 0 sehingga resistance tidak terhitung.)
    df["resistance_class"] = np.select(
        [df["resistance"].isna(), df["resistance"] < low_thresh_relaxed, df["resistance"] > high_thresh_relaxed],
        ["Unknown", "Very Low", "Very High"],
        default="Normal",
    )

    # ---- [Electrical fault handling] Sensor Fault - Electrical Readings ----
    # voltage_avr, current_avr, DAN power_total serentak 0 -> meteran listrik tidak
    # terbaca (bukan sekadar heater OFF, karena voltage suplai seharusnya tetap ada
    # walau heater dimatikan).
    electrical_sensor_fault_flag = ((df["voltage_avr"] == 0) & (df["current_avr"] == 0) &
                                     (df["power_total"] == 0))
    df["electrical_sensor_fault_flag"] = electrical_sensor_fault_flag.astype(int)
    electrical_sensor_fault_cond = electrical_sensor_fault_flag.fillna(False)

    # ---- [Electrical fault handling] Heater Open Circuit / Broken Element ----
    # Heater terbaca ON (heater1/heater2 > 0), voltage_avr tetap normal (bukan kasus
    # sensor fault di atas), tapi current_avr ~0 -> resistance melonjak (rangkaian
    # putus). Threshold 0.05A dipilih supaya TIDAK tumpang tindih dengan baris
    # Overheat asli pada data historis (current_avr terendah pada baris Overheat
    # sungguhan adalah ~0.12A).
    open_circuit_cond = ((df["heater_active"] == 1) & (df["voltage_avr"] > 0) &
                          (df["current_avr"] < 0.05) & ~electrical_sensor_fault_flag).fillna(False)

    # ---- Uncontrolled Heating ----
    uncontrolled_heating_cond = ((df["delta_temp_1h"] > 0) & (df["heater_both_off"] == 1)).fillna(False)

    # ---- Overheat / Low Temperature ----
    # [Aligned to real Notebook 2 output] Overheat is CRITICAL, not WARNING.
    overheat_cond = (mt > overheat_th).fillna(False)
    low_temp_cond = ((mt < low_th) & (mt >= LIQUIDUS_TEMP_C)).fillna(False)

    # ---- [Aligned to real Notebook 2 output] Sensor Fault — molten_temp == 0 ----
    sensor_fault_cond = (df["molten_temp_sensor_fault_flag"] == 1).fillna(False)

    conditions = [
        eutectic_cond, short_circuit_cond, open_circuit_cond, starvation_power_cond, mushy_zone_cond,
        overheat_cond, sensor_fault_cond, electrical_sensor_fault_cond,
        thermal_lag_only_cond, uncontrolled_heating_cond, low_temp_cond, voltage_abnormal_only_cond,
    ]
    reasons = [
        R_EUTECTIC, R_SHORT_CIRC, R_OPEN_CIRCUIT, R_STARVATION, R_MUSHY_ZONE, R_OVERHEAT,
        R_SENSOR_FAULT, R_ELEC_FAULT, R_THERMAL_LAG, R_UNCONTROLLED, R_LOW_TEMP, R_VOLTAGE_ABN,
    ]
    severities = [
        SEV_FATAL, SEV_FATAL, SEV_CRITICAL, SEV_CRITICAL, SEV_CRITICAL, SEV_CRITICAL,
        SEV_WARNING, SEV_WARNING, SEV_WARNING, SEV_WARNING, SEV_WARNING, SEV_WARNING,
    ]

    df["reason"] = np.select(conditions, reasons, default=R_NORMAL)
    df["severity_level"] = np.select(conditions, severities, default=SEV_NORMAL)
    df["is_anomaly"] = (df["severity_level"] != SEV_NORMAL).astype(int)
    return df


# ============================================================================
# 5. WEIGHTED ENSEMBLE SCORING
# ----------------------------------------------------------------------------
# Bobot   : Rule-Based 40% · XGBoost 40% · OCSVM 10% · LOF 10%
# Basis   : Rule-Based & XGBoost diberi bobot dominan karena precision tinggi
#           (rule = domain-deterministic, XGBoost = supervised dari label rule);
#           OCSVM & LOF (unsupervised) berperan sebagai sinyal pelengkap dengan
#           bobot lebih kecil karena lebih rentan false-positive pada data
#           operasional yang variatif.
# Skor    : weighted_score = 0.4*rule_flag + 0.4*P(anomaly|XGB) + 0.1*P(OCSVM) + 0.1*P(LOF)
#           [UPDATE #4] P(anomaly|XGB) sekarang = 1 - P(kelas NORMAL) dari output
#           multi-class softmax (4 kelas severity), bukan probabilitas biner lagi.
#           P(OCSVM)/P(LOF) diperoleh dari sigmoid(-decision_function) supaya
#           skor mentah (yang skalanya tak terbatas & tidak berupa probabilitas)
#           dipetakan proporsional ke rentang 0..1 (0 = sangat normal, 1 = sangat
#           anomali), konsisten dengan P(XGB) yang sudah berupa probabilitas.
# Threshold: weighted_score >= 0.50 -> "Anomali" (titik tengah skor gabungan).
#           SEBAGAI SAFETY INTERLOCK, rule_flag == 1 tetap men-trigger "Anomali"
#           secara langsung (hard override) sekalipun weighted_score < 0.50 —
#           sejalan dengan arahan domain knowledge bahwa Layer-1 rule bersifat
#           deterministik/safety-critical dan tidak boleh diserahkan sepenuhnya
#           ke model statistik (lihat Bagian D.1 dokumen domain knowledge).
# ============================================================================
W_RULE, W_XGB, W_OCSVM, W_LOF = 0.40, 0.40, 0.10, 0.10
SCORE_THRESHOLD = 0.50


def _sigmoid_anomaly_prob(decision_values, k=2.0):
    """decision_function sklearn: >0 cenderung inlier/normal, <0 cenderung outlier.
    Dipetakan ke pseudo-probabilitas anomali 0..1 lewat sigmoid(-k * decision)."""
    decision_values = np.asarray(decision_values, dtype=float)
    return 1.0 / (1.0 + np.exp(k * decision_values))


def compute_weighted_score(rule_flag, xgb_anomaly_proba, ocsvm_decision, lof_decision):
    """Versi vektor (batch) & skalar (live) — semua argumen bisa berupa array atau float.
    xgb_anomaly_proba = 1 - P(kelas NORMAL) dari model multi-class."""
    ocsvm_prob = _sigmoid_anomaly_prob(ocsvm_decision)
    lof_prob = _sigmoid_anomaly_prob(lof_decision)
    weighted_score = (W_RULE * np.asarray(rule_flag, dtype=float) +
                       W_XGB * np.asarray(xgb_anomaly_proba, dtype=float) +
                       W_OCSVM * ocsvm_prob +
                       W_LOF * lof_prob)
    final_flag = ((weighted_score >= SCORE_THRESHOLD) | (np.asarray(rule_flag) == 1)).astype(int)
    return weighted_score, final_flag, ocsvm_prob, lof_prob


def score_batch_weighted(df_labeled, metadata, scaler, ocsvm, lof, clf, feature_cols):
    """Menjalankan model ML pada dataset yang sudah melalui cleansing + feature
    engineering + rule labeling (batch), lalu menambahkan kolom skor ensemble."""
    df = df_labeled.copy()
    resistance_sentinel = metadata.get("resistance_sentinel", get_resistance_sentinel(metadata))
    df["resistance"] = df["resistance"].fillna(resistance_sentinel)

    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Fitur berikut tidak berhasil di-generate: {missing}")

    valid_mask = df[feature_cols].notna().all(axis=1)
    X = df.loc[valid_mask, feature_cols]

    X_scaled = scaler.transform(X)
    ocsvm_decision = ocsvm.decision_function(X_scaled)
    lof_decision = lof.decision_function(X_scaled)
    # [UPDATE #4] Multi-class XGBoost: P(anomaly) = 1 - P(kelas 0 / NORMAL)
    xgb_proba_matrix = clf.predict_proba(X)
    xgb_anomaly_proba = 1.0 - xgb_proba_matrix[:, 0]
    xgb_pred_class = xgb_proba_matrix.argmax(axis=1)

    weighted_score, final_flag, ocsvm_prob, lof_prob = compute_weighted_score(
        df.loc[valid_mask, "is_anomaly"].values, xgb_anomaly_proba, ocsvm_decision, lof_decision
    )

    df.loc[valid_mask, "ocsvm_anomaly_prob"] = ocsvm_prob
    df.loc[valid_mask, "lof_anomaly_prob"] = lof_prob
    df.loc[valid_mask, "xgb_anomaly_prob"] = xgb_anomaly_proba
    df.loc[valid_mask, "xgb_pred_severity"] = pd.Series(xgb_pred_class, index=df.loc[valid_mask].index).map(CLASS_TO_SEVERITY)
    df.loc[valid_mask, "weighted_score"] = weighted_score
    df.loc[valid_mask, "final_flag"] = final_flag
    df["final_flag"] = df["final_flag"].fillna(0).astype(int)
    df["weighted_score"] = df["weighted_score"].fillna(0.0)
    return df


# ============================================================================
# 6. LIVE FEATURE ENGINEERING (single reading, mirrors batch logic above)
# ============================================================================
def get_resistance_sentinel(metadata):
    if "resistance_sentinel" in metadata:
        return float(metadata["resistance_sentinel"])
    if "resistance_max" in metadata:
        return float(metadata["resistance_max"]) * 2.0
    return 2.0


def engineer_live_features(current_row, history_df, samples_per_hour, metadata):
    row = current_row.copy()

    # [UPDATE #1] Sensor fault handling — sama seperti versi batch.
    row["molten_temp_sensor_fault_flag"] = int(row["molten_temp"] == 0)
    row["molten_temp_valid"] = np.nan if row["molten_temp"] == 0 else row["molten_temp"]

    if row["current_avr"] == 0:
        row["resistance"] = get_resistance_sentinel(metadata)
    else:
        row["resistance"] = row["power_total"] / (row["current_avr"] ** 2)

    hist_valid_temp = None
    if len(history_df) > 0 and "molten_temp" in history_df.columns:
        hist_valid_temp = history_df["molten_temp"].replace(0, np.nan)

    if hist_valid_temp is not None and len(hist_valid_temp) > 0:
        idx_back = max(0, len(hist_valid_temp) - samples_per_hour)
        past_temp = hist_valid_temp.iloc[idx_back]
        row["delta_temp_1h"] = (row["molten_temp_valid"] - past_temp) if pd.notna(past_temp) and pd.notna(row["molten_temp_valid"]) else 0.0
    else:
        row["delta_temp_1h"] = 0.0

    row["heater_active"] = int((row["heater1"] > 0) or (row["heater2"] > 0))
    row["heater_both_off"] = int((row["heater1"] == 0) and (row["heater2"] == 0))

    win_short = pd.concat([history_df.tail(samples_per_hour - 1), pd.DataFrame([current_row])], ignore_index=True)
    row["current_avr_roll_std"] = win_short["current_avr"].std() if len(win_short) > 1 else 0.0
    row["power_total_roll_std"] = win_short["power_total"].std() if len(win_short) > 1 else 0.0

    win_long = pd.concat([history_df.tail(samples_per_hour * 3 - 1), pd.DataFrame([current_row])], ignore_index=True)
    if len(win_long) > 1 and win_long["current_avr"].std() > 0:
        spike_bound = win_long["current_avr"].mean() + 2 * win_long["current_avr"].std()
        row["current_avr_spike"] = bool(row["current_avr"] > spike_bound)
    else:
        row["current_avr_spike"] = False

    return row


def _check_thermal_lag_live(row, history_df, samples_per_hour, lag_delta=-5):
    """[UPDATE #3] Thermal Lag versi live: suhu (molten_temp_valid) harus TERUS
    turun tanpa rebound sama sekali di seluruh jendela penuh 1 jam terakhir
    (current reading + samples_per_hour-1 riwayat sebelumnya), heater harus
    aktif (ON) di sepanjang jendela tersebut, DAN net drop melewati lag_delta.
    lag_delta bertindak sebagai magnitude floor — tanpa ini, sinyal sensor yang
    naturally noisy (±1-3°C) dengan heater yang hampir selalu ON akan memicu
    rule ini terus-menerus meski bukan anomali sungguhan (divalidasi terhadap
    data real Notebook 2)."""
    if len(history_df) < samples_per_hour - 1:
        return False
    tail = history_df.tail(samples_per_hour - 1).copy()
    if "molten_temp" not in tail.columns:
        return False
    tail_valid_temp = tail["molten_temp"].replace(0, np.nan)
    tail_heater_active = ((tail.get("heater1", pd.Series(dtype=float)).fillna(0) > 0) |
                           (tail.get("heater2", pd.Series(dtype=float)).fillna(0) > 0)).astype(int) \
        if "heater1" in tail.columns and "heater2" in tail.columns else pd.Series([1] * len(tail))

    temps = pd.concat([tail_valid_temp, pd.Series([row["molten_temp_valid"]])], ignore_index=True)
    heaters = pd.concat([tail_heater_active, pd.Series([row["heater_active"]])], ignore_index=True)

    if temps.isna().any():
        return False
    diffs = temps.diff().dropna()
    no_rebound = bool((diffs <= 0).all())
    heater_all_on = bool((heaters == 1).all())
    net_drop = bool(temps.iloc[-1] - temps.iloc[0] <= lag_delta)
    return no_rebound and heater_all_on and net_drop


def apply_layer1_rules(row, history_df, metadata, ref_stats, samples_per_hour=4):
    """[UPDATE #2] Versi live/skalar dari Layer-1 rule engine — mengembalikan
    (reason, severity_level, is_anomaly, active_reasons) memakai prioritas yang
    sama persis dengan apply_layer1_rules_batch()."""
    overheat_th = metadata.get("overheat_threshold", 680)
    low_th = metadata.get("low_temp_threshold", 640)
    resistance_low_q10 = metadata.get("resistance_low_q10", ref_stats.get("resistance_low_q10"))
    resistance_high_q90 = metadata.get("resistance_high_q90", ref_stats.get("resistance_high_q90"))
    power_roll_std_q20 = metadata.get("power_roll_std_q20", ref_stats.get("power_roll_std_q20"))

    mt_valid = row["molten_temp_valid"]
    mt_ok = pd.notna(mt_valid)

    # ---- resistance_class untuk display/debug (Very Low/Normal/Very High/Unknown) ----
    if pd.isna(row["resistance"]):
        row["resistance_class"] = "Unknown"
    elif resistance_low_q10 is not None and row["resistance"] < resistance_low_q10:
        row["resistance_class"] = "Very Low"
    elif resistance_high_q90 is not None and row["resistance"] > resistance_high_q90:
        row["resistance_class"] = "Very High"
    else:
        row["resistance_class"] = "Normal"

    lag_delta = metadata.get("thermal_lag_delta", -5)
    thermal_lag_base = _check_thermal_lag_live(row, history_df, samples_per_hour, lag_delta=lag_delta)
    voltage_abnormal = bool(row["voltage_avr"] < VOLTAGE_MIN_V or row["voltage_avr"] > VOLTAGE_MAX_V)

    eutectic_cond = bool(mt_ok and mt_valid < EUTECTIC_TEMP_C)
    if resistance_low_q10 is not None and power_roll_std_q20 is not None and not pd.isna(row["resistance"]):
        short_circuit_cond = bool((row["resistance"] < resistance_low_q10) and
                                   (row["power_total_roll_std"] < power_roll_std_q20) and
                                   row.get("current_avr_spike", False))
    else:
        short_circuit_cond = False

    # ---- [Electrical fault handling] Sensor Fault - Electrical Readings ----
    # voltage_avr, current_avr, DAN power_total serentak 0 -> meteran listrik tidak
    # terbaca (bukan sekadar heater OFF, karena voltage suplai seharusnya tetap ada).
    electrical_sensor_fault_cond = bool(row["voltage_avr"] == 0 and row["current_avr"] == 0 and
                                         row["power_total"] == 0)

    # ---- [Electrical fault handling] Heater Open Circuit / Broken Element ----
    # Heater terbaca ON, voltage_avr tetap normal (bukan kasus sensor fault di atas),
    # tapi current_avr ~0 -> resistance melonjak (rangkaian putus). Threshold 0.05A
    # dipilih supaya TIDAK tumpang tindih dengan baris Overheat asli pada data
    # historis (current_avr terendah pada baris Overheat sungguhan adalah ~0.12A).
    open_circuit_cond = bool(row["heater_active"] == 1 and row["voltage_avr"] > 0 and
                              row["current_avr"] < 0.05 and not electrical_sensor_fault_cond)

    starvation_power_cond = bool(thermal_lag_base and voltage_abnormal)
    mushy_zone_cond = bool(mt_ok and EUTECTIC_TEMP_C <= mt_valid < LIQUIDUS_TEMP_C)
    thermal_lag_only_cond = bool(thermal_lag_base and not voltage_abnormal)
    uncontrolled_heating_cond = bool(row["delta_temp_1h"] > 0 and row["heater_both_off"] == 1)
    # [Aligned to real Notebook 2 output] Overheat is CRITICAL, not WARNING.
    overheat_cond = bool(mt_ok and mt_valid > overheat_th)
    low_temp_cond = bool(mt_ok and low_th > mt_valid >= LIQUIDUS_TEMP_C)
    voltage_abnormal_only_cond = bool(voltage_abnormal and not thermal_lag_base)
    # [Aligned to real Notebook 2 output] Sensor Fault — molten_temp == 0.
    sensor_fault_cond = bool(row.get("molten_temp_sensor_fault_flag", 0) == 1)

    ordered = [
        (eutectic_cond, R_EUTECTIC, SEV_FATAL),
        (short_circuit_cond, R_SHORT_CIRC, SEV_FATAL),
        (open_circuit_cond, R_OPEN_CIRCUIT, SEV_CRITICAL),
        (starvation_power_cond, R_STARVATION, SEV_CRITICAL),
        (mushy_zone_cond, R_MUSHY_ZONE, SEV_CRITICAL),
        (overheat_cond, R_OVERHEAT, SEV_CRITICAL),
        (sensor_fault_cond, R_SENSOR_FAULT, SEV_WARNING),
        (electrical_sensor_fault_cond, R_ELEC_FAULT, SEV_WARNING),
        (thermal_lag_only_cond, R_THERMAL_LAG, SEV_WARNING),
        (uncontrolled_heating_cond, R_UNCONTROLLED, SEV_WARNING),
        (low_temp_cond, R_LOW_TEMP, SEV_WARNING),
        (voltage_abnormal_only_cond, R_VOLTAGE_ABN, SEV_WARNING),
    ]

    active_reasons = [r for cond, r, sev in ordered if cond]
    top = next(((r, sev) for cond, r, sev in ordered if cond), (R_NORMAL, SEV_NORMAL))
    reason, severity_level = top
    is_anomaly = int(severity_level != SEV_NORMAL)

    rule_flags = {
        R_EUTECTIC: int(eutectic_cond), R_SHORT_CIRC: int(short_circuit_cond),
        R_OPEN_CIRCUIT: int(open_circuit_cond),
        R_STARVATION: int(starvation_power_cond), R_MUSHY_ZONE: int(mushy_zone_cond),
        R_OVERHEAT: int(overheat_cond), R_SENSOR_FAULT: int(sensor_fault_cond),
        R_ELEC_FAULT: int(electrical_sensor_fault_cond),
        R_THERMAL_LAG: int(thermal_lag_only_cond), R_UNCONTROLLED: int(uncontrolled_heating_cond),
        R_LOW_TEMP: int(low_temp_cond), R_VOLTAGE_ABN: int(voltage_abnormal_only_cond),
    }

    return reason, severity_level, is_anomaly, active_reasons, rule_flags


def compute_reference_stats(df_hist):
    ref = {}
    if df_hist is None or len(df_hist) == 0:
        return ref
    if "resistance" in df_hist.columns:
        ref["resistance_low_q10"] = float(df_hist["resistance"].dropna().quantile(0.10)) if df_hist["resistance"].notna().any() else None
        ref["resistance_high_q90"] = float(df_hist["resistance"].dropna().quantile(0.90)) if df_hist["resistance"].notna().any() else None
    if "power_total_roll_std" in df_hist.columns:
        ref["power_roll_std_q20"] = float(df_hist["power_total_roll_std"].quantile(0.20))
    return ref


# ============================================================================
# 7. GAUGE / CHART BUILDERS (Plotly, tema ungu — semua font mengikuti C_INK/C_MUTED
#    yang sudah dihitung otomatis dari tema aktif, lihat build_theme())
# ============================================================================
def make_gauge(value, title, val_range, unit, is_anomaly, zones=None):
    bar_color = C_DANGER if is_anomaly else C_PRIMARY
    steps = zones or [
        {"range": [val_range[0], val_range[1] * 0.8], "color": mix_hex(C_LIGHT, C_LILAC_BG, 0.65)},
        {"range": [val_range[1] * 0.8, val_range[1]], "color": mix_hex(C_LIGHT, C_LILAC_BG, 0.30)},
    ]
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        number={"suffix": f" {unit}", "font": {"color": C_INK, "size": 28}},
        title={"text": title, "font": {"size": 13, "color": C_MUTED}},
        gauge={
            "axis": {"range": val_range, "tickcolor": C_MUTED},
            "bar": {"color": bar_color, "thickness": 0.32},
            "bgcolor": C_CARD,
            "borderwidth": 0,
            "steps": steps,
            "threshold": {"line": {"color": C_INK, "width": 3}, "thickness": 0.8, "value": value},
        },
    ))
    fig.update_layout(height=230, margin=dict(l=20, r=20, t=40, b=10),
                       paper_bgcolor="rgba(0,0,0,0)", font=dict(color=C_INK),
                       hoverlabel=dict(bgcolor=C_CARD, font=dict(color=C_INK)))
    return fig


def make_mini_timeseries(hist, col, title, unit, anomaly_col="anomaly_flag"):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=hist[col], mode="lines", name=col,
                              line=dict(color=C_SECONDARY, width=2.5),
                              fill="tozeroy", fillcolor="rgba(128,90,213,0.12)"))
    if anomaly_col in hist.columns:
        anom = hist[hist[anomaly_col] == 1]
        if len(anom) > 0:
            fig.add_trace(go.Scatter(x=anom.index, y=anom[col], mode="markers", name="Anomali",
                                      marker=dict(color=C_DANGER, size=9, symbol="circle",
                                                  line=dict(color=C_CARD, width=1))))
    fig.update_layout(title=dict(text=title, font=dict(size=13, color=C_MUTED)),
                       height=230, margin=dict(l=10, r=10, t=35, b=10),
                       paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                       xaxis_title=None, yaxis_title=unit, showlegend=False,
                       font=dict(color=C_INK, size=11),
                       hoverlabel=dict(bgcolor=C_CARD, font=dict(color=C_INK)))
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridcolor=C_GRID)
    return fig


def make_full_timeseries(df_hist, col, title, anomaly_col="is_anomaly", growing=False):
    """[UPDATE #5] Visualisasi time-series dengan marker merah pada titik anomali.
    growing=True dipakai di mode Live (garis 'menyambung' seiring data baru
    ditambahkan ke session_state.history setiap kali operator menekan tombol
    'Muat Data Mesin'); growing=False dipakai di mode Upload untuk menampilkan
    tren penuh dataset sekaligus."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_hist["timestamp"], y=df_hist[col], mode="lines+markers" if growing else "lines",
                              name=col, line=dict(color=C_SECONDARY, width=1.8 if growing else 1.4),
                              marker=dict(size=4) if growing else None))
    if anomaly_col in df_hist.columns:
        anom = df_hist[df_hist[anomaly_col] == 1]
        if len(anom) > 0:
            fig.add_trace(go.Scatter(x=anom["timestamp"], y=anom[col], mode="markers", name="Anomali",
                                      marker=dict(color=C_DANGER, size=10 if growing else 6,
                                                  symbol="circle", line=dict(color=C_CARD, width=1))))
    fig.update_layout(title=title, height=420, paper_bgcolor="rgba(0,0,0,0)",
                       plot_bgcolor="rgba(0,0,0,0)", font=dict(color=C_INK),
                       legend=dict(orientation="h", y=1.1, font=dict(color=C_INK)),
                       hoverlabel=dict(bgcolor=C_CARD, font=dict(color=C_INK)))
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridcolor=C_GRID)
    return fig


def make_reason_bar(df_hist):
    """[UPDATE #5] Bar chart frekuensi `reason` (menggantikan RULE_COLS lama).
    Hanya menghitung baris anomali (reason != Normal Operation)."""
    counts = (df_hist.loc[df_hist["reason"] != R_NORMAL, "reason"]
              .value_counts()
              .reindex(REASON_ORDER)
              .fillna(0)
              .sort_values(ascending=True))
    fig = px.bar(counts, x=counts.values, y=counts.index, orientation="h",
                 color=counts.values, color_continuous_scale=PINK_SCALE,
                 labels={"x": "Jumlah Kejadian", "y": "Reason (Layer-1)"})
    fig.update_layout(height=360, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                       font=dict(color=C_INK), coloraxis_showscale=False,
                       title="Frekuensi Reason — Layer-1 Rule Engine",
                       hoverlabel=dict(bgcolor=C_CARD, font=dict(color=C_INK)))
    fig.update_xaxes(showgrid=True, gridcolor=C_GRID, color=C_INK)
    fig.update_yaxes(color=C_INK)
    return fig


def make_severity_bar(df_hist):
    """[UPDATE #5] Bar chart distribusi severity_level (baru)."""
    counts = df_hist["severity_level"].value_counts().reindex(SEVERITY_ORDER).fillna(0)
    colors = [SEVERITY_COLOR_MAP.get(s, C_MUTED) for s in counts.index]
    fig = go.Figure(go.Bar(x=counts.index, y=counts.values, marker_color=colors))
    fig.update_layout(height=360, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                       font=dict(color=C_INK), title="Distribusi Severity Level",
                       hoverlabel=dict(bgcolor=C_CARD, font=dict(color=C_INK)))
    fig.update_xaxes(showgrid=False, color=C_INK)
    fig.update_yaxes(showgrid=True, gridcolor=C_GRID, color=C_INK)
    return fig


def make_feature_importance(clf, feature_cols):
    if not hasattr(clf, "feature_importances_"):
        return None
    imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=True)
    fig = px.bar(imp, x=imp.values, y=imp.index, orientation="h",
                 color=imp.values, color_continuous_scale=PINK_SCALE,
                 labels={"x": "Importance", "y": "Fitur"})
    fig.update_layout(height=380, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                       font=dict(color=C_INK), coloraxis_showscale=False,
                       title="Feature Importance — XGBoost",
                       hoverlabel=dict(bgcolor=C_CARD, font=dict(color=C_INK)))
    fig.update_xaxes(showgrid=True, gridcolor=C_GRID, color=C_INK)
    fig.update_yaxes(color=C_INK)
    return fig


# ============================================================================
# 8. APP START
# ============================================================================

# ---- Pilih tema tampilan (harus paling awal, sebelum inject_css()) --------
with st.sidebar:
    st.markdown(f"<h2 style='color:{C_PRIMARY}; margin-bottom:0;'>🌸 Predictive Maintenance</h2>", unsafe_allow_html=True)

    st.markdown("### 🎨 Tema Tampilan")
    theme_name = st.selectbox(
        "Tema", list(PRESET_BASE.keys()) + ["🎛️ Custom"],
        label_visibility="collapsed",
    )
    if theme_name == "🎛️ Custom":
        cc1, cc2 = st.columns(2)
        with cc1:
            _primary = st.color_picker("Primary", "#EC4899")
            _light = st.color_picker("Aksen muda", "#FBCFE8")
            _bg = st.color_picker("Background", "#FFF0F6")
        with cc2:
            _secondary = st.color_picker("Secondary", "#F472B6")
            _ink = st.color_picker("Warna teks", "#6B1D42")
            _card = st.color_picker("Warna card", "#FFFFFF")
        st.caption("Warna muted/border/grid/status dihitung otomatis dari 6 warna "
                   "di atas supaya tulisan tetap terbaca di kombinasi apa pun.")
        theme = build_theme(_primary, _secondary, _light, _bg, _card, _ink)
    else:
        theme = build_theme(**PRESET_BASE[theme_name])

# ---- Terapkan tema terpilih ke seluruh konstanta warna global -------------
C_PRIMARY, C_SECONDARY, C_LIGHT = theme["primary"], theme["secondary"], theme["light"]
C_LILAC_BG, C_CARD, C_INK, C_MUTED = theme["bg"], theme["card"], theme["ink"], theme["muted"]
C_BORDER, C_GRID, C_SIDEBAR_BG = theme["border"], theme["grid"], theme["sidebar_bg"]
C_DANGER, C_WARNING, C_SUCCESS = theme["danger"], theme["warning"], theme["success"]
C_HEADER_TEXT, C_IS_DARK = theme["header_text"], theme["is_dark"]
PINK_SCALE = build_pink_scale(C_LILAC_BG, C_PRIMARY)

# Warna badge per severity level, mengikuti tema aktif.
SEVERITY_COLOR_MAP = {
    SEV_NORMAL: C_SUCCESS,
    SEV_WARNING: C_WARNING,
    SEV_CRITICAL: mix_hex(C_WARNING, C_DANGER, 0.6),
    SEV_FATAL: C_DANGER,
}

inject_css()

if "history" not in st.session_state:
    st.session_state.history = pd.DataFrame(columns=CORE_FEATURES + ["timestamp", "anomaly_flag", "weighted_score"])
if "machine_on" not in st.session_state:
    st.session_state.machine_on = True
if "demo_mode" not in st.session_state:
    st.session_state.demo_mode = False
if "uploaded_result" not in st.session_state:
    st.session_state.uploaded_result = None

# ---- Load artifacts (real -> fallback demo) --------------------------------
models_loaded, load_error = True, None
try:
    ocsvm, lof, clf, scaler, metadata = load_artifacts(MODEL_PATH)
    if "feature_cols" not in metadata:
        raise KeyError("model_metadata.json tidak memiliki key 'feature_cols'.")
    FEATURE_COLS = list(metadata["feature_cols"])
    df_hist = None
    try:
        df_hist = load_historical_data(DATA_PATH)
    except Exception:
        df_hist = None
except Exception as e:
    models_loaded, load_error = False, str(e)
    FEATURE_COLS, df_hist = None, None

if not models_loaded:
    st.session_state.demo_mode = True
    ocsvm, lof, clf, scaler, metadata, df_hist = build_demo_artifacts()
    FEATURE_COLS = list(metadata["feature_cols"])

if len(FEATURE_COLS) != EXPECTED_N_FEATURES:
    st.warning(f"⚠️ Jumlah fitur pada metadata ({len(FEATURE_COLS)}) berbeda dari yang "
               f"diharapkan ({EXPECTED_N_FEATURES}). Dashboard tetap memakai daftar dari metadata.")

ref_stats = compute_reference_stats(df_hist)

# ============================================================================
# 9. SIDEBAR — Dual Mode Input
# ============================================================================
with st.sidebar:
    if st.session_state.demo_mode:
        st.caption("🧪 Mode Demo aktif — artifact Kaggle asli tidak ditemukan, memakai data & model sintetis.")

    st.markdown("### 📥 Mode Input Data")
    input_mode = st.radio(
        "Sumber data", ["📡 Live dari Mesin", "📁 Upload File (CSV/XLSX)"],
        label_visibility="collapsed",
    )

    st.markdown("### Identitas Mesin")
    nama_mesin = st.text_input("Nama Mesin", "Holding Furnace HPDC 15")
    tipe_mesin = st.selectbox("Tipe", ["Immersion Aluminium Holding Furnace (1120 kg/ch)"])
    line_prod = st.selectbox("Line Produksi (Casting Operations)", ["Plant 3", "Plant 3a", "Plant 5"])
    shift_operator = st.selectbox("Shift Operator", ["Cia (Shift A)", "Alex (Shift B)", "Agnes (Shift C)"])

    if input_mode == "📡 Live dari Mesin":
        st.markdown("### Pengaturan Mesin")
        monitoring_on = st.toggle("Monitoring Sensor (real-time stream)", value=True)
        ai_detection_on = st.toggle("AI Anomaly Detection (Rule + Weighted Ensemble)", value=True)
        alert_on = st.toggle("Alert Notifikasi (email + buzzer)", value=True)

        st.markdown("### Simulasi Sensor Live")
        interval_menit = st.number_input("Interval antar pembacaan (menit)", 1, 30, 15)
        samples_per_hour = max(1, 60 // interval_menit)

        # ---- [UPDATE #6] Slider dengan tooltip fisik (help=) ----
        # Batas bawah diset ke 0.0 (bukan 580/300) supaya SEMUA state "OFF"/"fault"
        # yang dipakai oleh rule engine (molten_temp==0, heater1/2==0, voltage_avr==0)
        # bisa benar-benar disimulasikan lewat slider, bukan hanya lewat upload data.
        molten_temp = st.slider(
            "molten_temp (°C)", 0.0, 720.0, 652.0, step=0.5,
            help=("Rentang operasional normal: 640-660°C. Di bawah 615°C logam memasuki "
                  "Mushy Zone (fase transisi cair-padat, diagram fasa Al-Si). Di bawah "
                  "592°C (Eutectic) risiko pembekuan total. Nilai TEPAT 0°C dibaca "
                  "sebagai **sensor fault** (thermocouple tidak terbaca), bukan suhu asli."),
        )
        current_avr = st.slider(
            "current_avr (A)", 0.0, 500.0, 118.0, step=1.0,
            help=("Arus rata-rata pada elemen heater. Ikuti Hukum Ohm bersama voltage_avr & "
                  "power_total: jika arus mendekati 0 sementara tegangan tetap ada, resistance "
                  "melonjak ke rentang Megaohm — indikasi heater putus/mati (open circuit)."),
        )
        power_total = st.slider(
            "power_total (kW)", 0.0, 200.0, 44.0, step=0.5,
            help=("Daya total yang terukur pada heater. Bersama current_avr membentuk "
                  "resistance = power_total / current_avr². Daya yang stagnan sementara "
                  "arus melonjak adalah tanda khas Short Circuit / Heating Element Failure."),
        )
        heater1 = st.slider("heater1 (°C)", 0.0, 1200.0, 680.0, step=0.5,
                             help="Suhu elemen Immersion Heater #1. Nilai 0 dibaca sebagai heater OFF.")
        heater2 = st.slider("heater2 (°C)", 0.0, 1200.0, 675.0, step=0.5,
                             help="Suhu elemen Immersion Heater #2. Nilai 0 dibaca sebagai heater OFF.")
        voltage_avr = st.slider(
            "voltage_avr (V)", 0.0, 450.0, 380.0, step=1.0,
            help=("Tegangan nominal suplai heater adalah 380V. Fluktuasi di luar ±10% "
                  "(di bawah 342V atau di atas 418V) secara fisik tidak normal dan akan "
                  "memicu peringatan 'Starvation Power / Voltage Drop', terutama bila "
                  "terjadi bersamaan dengan Thermal Lag. Nilai TEPAT 0V (bersamaan dengan "
                  "current_avr & power_total 0) dibaca sebagai **sensor fault kelistrikan**."),
        )


        # ---- [Electrical fault handling] Resistance = nilai TURUNAN, bukan sensor
        # independen — karena itu tidak ada slider terpisah untuknya. Nilainya
        # otomatis mengikuti current_avr & power_total di atas (resistance =
        # power_total / current_avr²) dan ditampilkan live di sini + di tab
        # 'Live Monitoring' (Parameter Real-Time & Status AI).
        _live_resistance = (float("inf") if current_avr == 0 else power_total / (current_avr ** 2))
        _resistance_display = "∞ (Open Circuit)" if current_avr == 0 else f"{_live_resistance:.4f} Ω"
        st.caption(f"⚡ **Resistance):** {_resistance_display} — dihitung otomatis "
                   "dari power_total / current_avr² (Hukum Ohm). Geser "
                   "current_avr mendekati 0 (dengan heater ON & voltage tetap ada) untuk mensimulasikan "
                   "**Heater Open Circuit / Broken Element**; geser current_avr, power_total, DAN "
                   "voltage_avr semuanya ke 0 untuk mensimulasikan **Sensor Fault - Electrical Readings**.")

        st.info("💡 Tip: Jika Heater 1 & 2 diset tinggi tetapi `molten_temp` digeser turun "
                "perlahan selama beberapa menit (tanpa rebound sama sekali dalam 1 jam), AI "
                "akan mendeteksi anomali **Thermal Lag** — indikasi heater ON namun gagal "
                "menghantarkan panas ke logam.")

        col_a, col_b = st.columns(2)
        feed_btn = col_a.button("▶ Muat Data Mesin", use_container_width=True)
        reset_btn = col_b.button("↺ Reset", use_container_width=True)

        st.markdown("---")
        toggle_label = "🔴 Matikan Mesin" if st.session_state.machine_on else "🟢 Nyalakan Mesin"
        if st.button(toggle_label, use_container_width=True):
            st.session_state.machine_on = not st.session_state.machine_on
    else:
        # Placeholder default supaya variabel tetap terdefinisi saat mode Upload aktif
        monitoring_on = ai_detection_on = alert_on = True
        samples_per_hour = 4
        reset_btn = False
        st.markdown("### Mode Upload Aktif")
        st.caption("Gunakan panel utama untuk meng-unggah file CSV/XLSX berisi data mentah "
                   "(kolom wajib: timestamp, molten_temp, heater1, heater2, voltage_avr, "
                   "current_avr, power_total). Preprocessing berjalan otomatis di backend.")

    st.markdown("### Info Mesin")
    st.caption(f"**Nama:** {nama_mesin}  \n**Line:** {line_prod}  \n**Operator:** {shift_operator}  \n"
               f"**Mode:** {input_mode}")

if input_mode == "📡 Live dari Mesin" and reset_btn:
    st.session_state.history = st.session_state.history.iloc[0:0]

# ============================================================================
# 10. HEADER — Waktu Lokal Asia/Jakarta (WIB), real-time
# ============================================================================
now_str = now_wib().strftime("%d %b %Y · %H:%M:%S WIB")
st.markdown(f"""
<div class="app-header">
    <h1>🌸 Holding Furnace Predictive Maintenance</h1>
    <p>{line_prod} · Immersion Aluminium Holding Furnace (1120 kg/ch) · Last sync: {now_str}</p>
</div>
""", unsafe_allow_html=True)

# ============================================================================
# 11. MODE: LIVE DARI MESIN
# ============================================================================
if input_mode == "📡 Live dari Mesin":
    MACHINE_ON = st.session_state.machine_on

    if not MACHINE_ON:
        st.markdown("""<div class="off-box">⏻ Mesin dalam kondisi <b>OFF</b> — sinyal sensor tidak tersedia.
                    Aktifkan power mesin di sidebar untuk memulai monitoring dan deteksi anomali AI real-time.</div>""",
                    unsafe_allow_html=True)
        cols = st.columns(4)
        labels = [("Molten Temp", "°C"), ("Current (Arus)", "A"), ("Power Total", "kW"), ("Weighted Score", "")]
        for c, (lbl, unit) in zip(cols, labels):
            with c:
                metric_card(lbl, "—", unit, sub="Tidak ada sinyal", status="normal")
        st.markdown("<div class='card' style='text-align:center; padding:60px 0;'>"
                    f"<div style='font-size:42px;'>⏻</div>"
                    f"<h4 style='color:{C_MUTED}; margin-top:10px;'>AI Monitoring Nonaktif</h4>"
                    "<p style='color:#8B84A6;'>Nyalakan mesin untuk memulai deteksi anomali secara real-time.</p></div>",
                    unsafe_allow_html=True)
        st.stop()

    # ---- 11a. LIVE FEATURE ENGINEERING + WEIGHTED PREDICTION ----
    current_input = pd.Series({
        "molten_temp": molten_temp, "heater1": heater1, "heater2": heater2,
        "voltage_avr": voltage_avr, "current_avr": current_avr, "power_total": power_total,
    })

    live_row = engineer_live_features(current_input, st.session_state.history, samples_per_hour, metadata)
    reason, severity_level, rule_flag, active_reasons, rule_flags = apply_layer1_rules(
        live_row, st.session_state.history, metadata, ref_stats, samples_per_hour
    )

    missing_cols = [c for c in FEATURE_COLS if c not in live_row.index]
    if missing_cols:
        st.error(f"Fitur berikut ada di metadata['feature_cols'] tapi tidak berhasil di-generate: {missing_cols}")
        st.stop()

    X_live = pd.DataFrame([live_row[FEATURE_COLS].values], columns=FEATURE_COLS)
    if X_live.isna().any().any():
        st.error(f"Ditemukan NaN pada fitur: {X_live.columns[X_live.isna().any()].tolist()} sebelum scaling.")
        st.stop()

    if ai_detection_on:
        X_live_scaled = scaler.transform(X_live)
        ocsvm_decision = float(ocsvm.decision_function(X_live_scaled)[0])
        lof_decision = float(lof.decision_function(X_live_scaled)[0])
        ocsvm_flag = int(ocsvm.predict(X_live_scaled)[0] == -1)
        lof_flag = int(lof.predict(X_live_scaled)[0] == -1)
        # [UPDATE #4] Prediksi multi-class: array berisi [P(NORMAL), P(WARNING), P(CRITICAL), P(FATAL)]
        clf_proba_array = clf.predict_proba(X_live)[0]
        clf_anomaly_proba = 1.0 - float(clf_proba_array[0])  # 1 - P(NORMAL)
        clf_class = int(np.argmax(clf_proba_array))          # 0..3
        clf_pred_severity = CLASS_TO_SEVERITY.get(clf_class, SEV_NORMAL)
        clf_flag = int(clf_class > 0)
        ml_votes = ocsvm_flag + lof_flag + clf_flag

        weighted_score, final_flag_arr, ocsvm_prob, lof_prob = compute_weighted_score(
            rule_flag, clf_anomaly_proba, ocsvm_decision, lof_decision
        )
        anomaly_score = round(float(weighted_score), 3)
        final_flag = int(final_flag_arr)
    else:
        ocsvm_flag = lof_flag = clf_flag = ml_votes = 0
        clf_anomaly_proba = 0.0
        clf_class = 0
        clf_pred_severity = SEV_NORMAL
        ocsvm_prob = lof_prob = 0.0
        final_flag = rule_flag
        anomaly_score = round(W_RULE, 3) if rule_flag else 0.0

    # ---- Estimasi waktu-ke-ambang batas (ekstrapolasi laju perubahan suhu) ----
    eta_text = "—"
    if live_row["delta_temp_1h"] != 0:
        rate_per_min = live_row["delta_temp_1h"] / max(samples_per_hour, 1)
        overheat_th = metadata.get("overheat_threshold", 680)
        low_th = metadata.get("low_temp_threshold", 640)
        if rate_per_min > 0 and molten_temp < overheat_th:
            minutes = (overheat_th - molten_temp) / rate_per_min * interval_menit
            if 0 < minutes < 600:
                eta_text = f"~{minutes:.0f} menit menuju Overheat"
        elif rate_per_min < 0 and molten_temp > low_th:
            minutes = (molten_temp - low_th) / abs(rate_per_min) * interval_menit
            if 0 < minutes < 600:
                eta_text = f"~{minutes:.0f} menit menuju Low Temp"

    # ---- [UPDATE #4/#5] TOP ACTIONABLE ALERT ----
    if final_flag == 1:
        actionable_alert(
            active_reasons,
            extra_note=(f"Severity: {severity_level} · Weighted anomaly score: {anomaly_score:.3f} "
                        f"(threshold ≥ {SCORE_THRESHOLD:.2f}) · "
                        f"{'Estimasi ' + eta_text + '.' if eta_text != '—' else ''} "
                        f"Confidence XGBoost: {clf_anomaly_proba*100:.1f}%.")
        )

    # ---- TABS ----
    tab_live, tab_hist, tab_debug = st.tabs(
        ["📈 Live Monitoring & AI Prediction", "📊 Analitik Historis", "🧠 Model & Debug"]
    )

    with tab_live:
        k1, k2, k3, k4 = st.columns(4)
        with k1:
            temp_status = "danger" if rule_flags.get(R_OVERHEAT) or rule_flags.get(R_LOW_TEMP) or severity_level in (SEV_CRITICAL, SEV_FATAL) else "normal"
            temp_sub = (
                "🚨 Sensor Fault (0°C)" if live_row.get("molten_temp_sensor_fault_flag") else
                f"⚠ {reason}" if severity_level != SEV_NORMAL and reason in (R_EUTECTIC, R_MUSHY_ZONE, R_OVERHEAT, R_LOW_TEMP) else
                "✓ Normal"
            )
            metric_card("Molten Temp", f"{molten_temp:.1f}", "°C", sub=temp_sub,
                        status="danger" if temp_status == "danger" else "ok")
        with k2:
            current_danger = bool(live_row.get("current_avr_spike") or rule_flags.get(R_OPEN_CIRCUIT) or
                                   rule_flags.get(R_ELEC_FAULT) or rule_flags.get(R_SHORT_CIRC))
            current_sub = (
                "🚨 Open Circuit (heater putus)" if rule_flags.get(R_OPEN_CIRCUIT) else
                "🚨 Sensor tidak terbaca" if rule_flags.get(R_ELEC_FAULT) else
                "🚨 Short Circuit" if rule_flags.get(R_SHORT_CIRC) else
                "⚠ Lonjakan arus" if live_row.get("current_avr_spike") else
                "✓ Normal"
            )
            metric_card("Current (Arus)", f"{current_avr:.1f}", "A", sub=current_sub,
                        status="danger" if current_danger else "ok")
        with k3:
            power_danger = bool(rule_flags.get(R_OPEN_CIRCUIT) or rule_flags.get(R_ELEC_FAULT) or
                                 rule_flags.get(R_SHORT_CIRC))
            resistance_class = live_row.get("resistance_class", "Normal")
            power_sub = (
                f"⚠ Resistance {resistance_class}" if resistance_class in ("Very High", "Very Low") else
                "✓ Normal"
            )
            metric_card("Power Total", f"{power_total:.1f}", "kW", sub=power_sub,
                        status="danger" if power_danger else "ok")
        with k4:
            metric_card("Weighted Anomaly Score", f"{anomaly_score:.3f}", "",
                        sub=f"🚨 {severity_level}" if final_flag else "✓ Kondisi Normal",
                        status="danger" if final_flag else "ok")

        g1, g2, g3 = st.columns(3)
        with g1:
            st.plotly_chart(make_gauge(molten_temp, "Molten Temperature", [580, 720], "°C",
                                        severity_level != SEV_NORMAL,
                                        zones=[{"range": [580, EUTECTIC_TEMP_C], "color": mix_hex(C_DANGER, C_LILAC_BG, 0.65)},
                                               {"range": [EUTECTIC_TEMP_C, LIQUIDUS_TEMP_C], "color": mix_hex(C_WARNING, C_LILAC_BG, 0.55)},
                                               {"range": [LIQUIDUS_TEMP_C, metadata.get("low_temp_threshold", 640)], "color": mix_hex(C_LIGHT, C_LILAC_BG, 0.55)},
                                               {"range": [metadata.get("low_temp_threshold", 640), metadata.get("overheat_threshold", 680)], "color": mix_hex(C_LIGHT, C_LILAC_BG, 0.30)},
                                               {"range": [metadata.get("overheat_threshold", 680), 720], "color": mix_hex(C_DANGER, C_LILAC_BG, 0.80)}]),
                            use_container_width=True)
        with g2:
            st.plotly_chart(make_gauge(current_avr, "Current (Arus)", [0, 500], "A", bool(live_row.get("current_avr_spike"))), use_container_width=True)
        with g3:
            st.plotly_chart(make_gauge(power_total, "Power Total", [0, 200], "kW", False), use_container_width=True)

        # ---- Animasi time-series (growing) + riwayat + download ----
        st.markdown("#### Riwayat Sesi — Time-Series Real-Time (menyambung)")
        if feed_btn:
            new_row = current_input.copy()
            new_row["timestamp"] = now_wib().replace(tzinfo=None)
            new_row["anomaly_flag"] = final_flag
            new_row["weighted_score"] = anomaly_score
            new_row["severity_level"] = severity_level
            new_row["reason"] = reason
            st.session_state.history = pd.concat(
                [st.session_state.history, pd.DataFrame([new_row])], ignore_index=True
            ).tail(300)

        hist = st.session_state.history.reset_index(drop=True)
        if len(hist) > 0:
            st.plotly_chart(
                make_full_timeseries(hist, "molten_temp", "molten_temp — Riwayat Sesi Live (titik merah = anomali)",
                                      anomaly_col="anomaly_flag", growing=True),
                use_container_width=True,
            )
            c1, c2, c3 = st.columns(3)
            with c1:
                st.plotly_chart(make_mini_timeseries(hist, "current_avr", "Current (A)", "A"), use_container_width=True)
            with c2:
                st.plotly_chart(make_mini_timeseries(hist, "power_total", "Power Total (kW)", "kW"), use_container_width=True)
            with c3:
                st.plotly_chart(make_mini_timeseries(hist, "weighted_score", "Weighted Score", ""), use_container_width=True)

            # 5 baris terakhir + tombol download log
            st.markdown("##### 5 Baris Terakhir (Log Data Ter-label AI)")
            st.dataframe(hist.tail(5), use_container_width=True, hide_index=True)
            st.download_button(
                "⬇ Download Log Sesi (CSV)",
                data=hist.to_csv(index=False).encode("utf-8"),
                file_name=f"live_session_log_{now_wib().strftime('%Y%m%d_%H%M%S')}.csv",
                mime="text/csv",
                use_container_width=True,
            )
        else:
            st.info("Klik **'▶ Muat Data Mesin'** di sidebar untuk mulai membangun riwayat sesi.")

        st.markdown("#### Parameter Real-Time & Status AI")
        p1, p2 = st.columns([1.1, 1])
        with p1:
            st.markdown('<div class="card"><h4>Parameter Real-Time</h4>', unsafe_allow_html=True)
            progress_row("Molten Temp", molten_temp, 720, "°C", C_PRIMARY)
            progress_row("Current (avg)", current_avr, 500, "A", C_SECONDARY)
            progress_row("Power Total", power_total, 200, "kW", C_SECONDARY)
            progress_row("Resistance", live_row["resistance"], max(live_row["resistance"] * 1.5, 1), "Ω", C_LIGHT)
            st.caption(f"Resistance class: **{live_row.get('resistance_class', 'Normal')}** "
                       "(Very Low → risiko Short Circuit · Very High → risiko Open Circuit/heater putus)")
            progress_row("Delta Temp (1h, valid)", live_row["delta_temp_1h"], max(abs(live_row["delta_temp_1h"]) * 1.5, 5), "°C", C_LIGHT)
            progress_row("Current Roll-Std", live_row["current_avr_roll_std"], max(live_row["current_avr_roll_std"] * 1.5, 5), "A", C_LIGHT)
            st.markdown('</div>', unsafe_allow_html=True)

        with p2:
            st.markdown('<div class="card"><h4>Status AI & Prediksi</h4>', unsafe_allow_html=True)
            severity_badge(severity_level)
            st.write("")
            st.write(f"**Reason (top priority):** {reason}")
            st.write(f"**Weighted score:** {anomaly_score:.3f}  (threshold ≥ {SCORE_THRESHOLD:.2f})")
            st.write(f"**Waktu ke ambang batas:** {eta_text}")
            st.write(f"**Model aktif:** Rule(40%) + XGBoost(40%) + OCSVM(10%) + LOF(10%)")
            st.write(f"**XGBoost predicted severity:** {clf_pred_severity} (confidence anomaly: {clf_anomaly_proba*100:.1f}%)")
            if live_row.get("molten_temp_sensor_fault_flag"):
                st.warning("⚠️ molten_temp == 0 terdeteksi — dibaca sebagai **sensor fault**, "
                           "bukan temperature drop asli. Fitur delta_temp_1h & rule threshold "
                           "mengabaikan pembacaan ini (memakai molten_temp_valid).")
            if rule_flags.get(R_ELEC_FAULT):
                st.warning("⚠️ voltage_avr, current_avr, DAN power_total serentak 0 — dibaca sebagai "
                           "**meteran listrik tidak terbaca** (bukan heater OFF, karena voltage suplai "
                           "seharusnya tetap ~380V meski heater dimatikan).")
            if rule_flags.get(R_OPEN_CIRCUIT):
                st.warning("⚠️ current_avr mendekati 0A padahal voltage_avr normal & heater terbaca ON — "
                           "resistance melonjak ekstrem, indikasi **elemen heater putus (open circuit)**.")

            st.markdown("**Rule Layer-1 (12 rule fisik):**")
            chips = "".join(
                f'<span class="rule-chip {"rule-on" if v else "rule-off"}">{k}</span>'
                for k, v in rule_flags.items()
            )
            st.markdown(chips, unsafe_allow_html=True)

            st.markdown("**Kontribusi Skor Weighted Ensemble:**")
            st.write(f"- Rule-Based (40%): {'🔴' if rule_flag else '🟢'} kontribusi {W_RULE*rule_flag:.3f}")
            st.write(f"- XGBoost (40%): p={clf_anomaly_proba:.2f} → kontribusi {W_XGB*clf_anomaly_proba:.3f}")
            st.write(f"- OCSVM (10%): p={ocsvm_prob:.2f} → kontribusi {W_OCSVM*ocsvm_prob:.3f}")
            st.write(f"- LOF (10%): p={lof_prob:.2f} → kontribusi {W_LOF*lof_prob:.3f}")
            st.write(f"- **Total weighted score: {anomaly_score:.3f}**")

            if active_reasons:
                rec = " · ".join(active_reasons[:2])
                st.markdown(f'<div class="rec-box">💡 <b>Rekomendasi:</b> {rec}. Segera jadwalkan inspeksi & konfirmasi manual pada panel mesin.</div>', unsafe_allow_html=True)
            elif final_flag:
                st.markdown('<div class="rec-box">💡 <b>Rekomendasi:</b> Skor ensemble ML melewati ambang batas meski rule belum menyala — pantau tren 15-30 menit ke depan.</div>', unsafe_allow_html=True)
            else:
                st.markdown('<div class="rec-box">✅ Tidak ada tindakan diperlukan. Operasi berjalan dalam rentang normal.</div>', unsafe_allow_html=True)
            st.markdown('</div>', unsafe_allow_html=True)

    # ------------------------------------------------------------- TAB HISTORIS
    with tab_hist:
        if df_hist is None or len(df_hist) == 0:
            st.info("Data historis (`labeled_data.csv`) tidak ditemukan di `DATA_PATH`. "
                    "Analitik historis membutuhkan output Notebook 2.")
        else:
            st.caption(f"Dataset historis: {len(df_hist):,} baris · "
                       f"{int(df_hist['is_anomaly'].sum()):,} anomali "
                       f"({df_hist['is_anomaly'].mean()*100:.2f}%)")

            st.plotly_chart(make_full_timeseries(df_hist, "molten_temp", "Timeline molten_temp — Anomali Ditandai (Layer-1 Rules)",
                                                  anomaly_col="is_anomaly"),
                             use_container_width=True)

            bc1, bc2 = st.columns(2)
            with bc1:
                if "reason" in df_hist.columns:
                    st.plotly_chart(make_reason_bar(df_hist), use_container_width=True)
                else:
                    st.info("Kolom `reason` tidak tersedia pada data historis (data lama / belum di-refresh Notebook 2).")
            with bc2:
                if "severity_level" in df_hist.columns:
                    st.plotly_chart(make_severity_bar(df_hist), use_container_width=True)
                else:
                    fig_imp = make_feature_importance(clf, FEATURE_COLS)
                    if fig_imp is not None:
                        st.plotly_chart(fig_imp, use_container_width=True)

            fig_imp = make_feature_importance(clf, FEATURE_COLS)
            if fig_imp is not None:
                st.plotly_chart(fig_imp, use_container_width=True)

    # ------------------------------------------------------------------ TAB DEBUG
    with tab_debug:
        st.markdown("#### Detection Breakdown")
        breakdown_df = pd.DataFrame({
            "Komponen": ["Layer-1 Rule Engine (40%)", "XGBoost (40%)", "OCSVM (10%)", "LOF (10%)", "Weighted Ensemble"],
            "Hasil": [
                f"{severity_level} — {reason}" if rule_flag else "NORMAL",
                f"Predicted: {clf_pred_severity} (p_anomaly={clf_anomaly_proba:.3f})" if clf_flag else f"Normal (p_anomaly={clf_anomaly_proba:.3f})",
                f"Anomaly (p={ocsvm_prob:.3f})" if ocsvm_flag else f"Normal (p={ocsvm_prob:.3f})",
                f"Anomaly (p={lof_prob:.3f})" if lof_flag else f"Normal (p={lof_prob:.3f})",
                f"🚨 ANOMALY (score={anomaly_score:.3f})" if final_flag else f"✅ NORMAL (score={anomaly_score:.3f})",
            ],
        })
        st.dataframe(breakdown_df, use_container_width=True, hide_index=True)

        if active_reasons:
            st.markdown("**Active rule reasons (urutan prioritas):**")
            for r in active_reasons:
                st.write(f"⚠️ {r}")

        st.markdown("#### Feature Vector Terkirim ke Model")
        st.dataframe(X_live, use_container_width=True)

        st.markdown("#### Model Metadata")
        st.json(metadata)

        st.markdown("#### Reference Stats (dari data historis, untuk rule dinamis)")
        st.json(ref_stats if ref_stats else {"info": "Tidak ada data historis dimuat — memakai fallback metadata/default."})

    st.caption("Dashboard merekonstruksi feature engineering Notebook 2 secara live (termasuk sensor-fault "
               "handling molten_temp_valid), menerapkan Layer-1 rule engine berbasis severity (NORMAL/WARNING/"
               "CRITICAL/FATAL) lewat np.select, dan menggabungkannya dengan Weighted Ensemble "
               "(Rule 40% · XGBoost multi-class 40% · OCSVM 10% · LOF 10%).")

# ============================================================================
# 12. MODE: UPLOAD FILE (CSV/XLSX)
# ============================================================================
else:
    tab_upload, tab_hist, tab_debug = st.tabs(
        ["📁 Upload & Prediksi AI", "📊 Analitik Historis", "🧠 Model & Debug"]
    )

    with tab_upload:
        st.markdown("#### 1️⃣ Upload Data Mentah")
        st.caption("Kolom wajib: `timestamp` (atau `date`/`datetime`), `molten_temp`, `heater1`, `heater2`, "
                   "`voltage_avr`, `current_avr`, `power_total`.")
        uploaded_file = st.file_uploader("Pilih file CSV atau XLSX", type=["csv", "xlsx", "xls"])

        if uploaded_file is not None:
            try:
                if uploaded_file.name.lower().endswith((".xlsx", ".xls")):
                    df_raw_upload = pd.read_excel(uploaded_file)
                else:
                    df_raw_upload = pd.read_csv(uploaded_file)
            except Exception as e:
                st.error(f"Gagal membaca file: {e}")
                df_raw_upload = None

            if df_raw_upload is not None:
                # ---- Tampilkan RAW DATA terlebih dahulu ----
                st.markdown("#### 2️⃣ Raw Data (Sebelum Preprocessing)")
                st.dataframe(df_raw_upload.head(20), use_container_width=True, hide_index=True)
                st.caption(f"Total baris raw: {len(df_raw_upload):,}")

                # ---- Backend preprocessing otomatis: cleansing -> feature engineering -> rules -> ML ----
                try:
                    upload_samples_per_hour = max(1, 60 // 15)
                    with st.spinner("Menjalankan preprocessing otomatis (cleansing + feature engineering)..."):
                        df_clean, cleaning_log = cleanse_raw_data(df_raw_upload)
                        df_feat = engineer_batch_features(df_clean, samples_per_hour=upload_samples_per_hour)
                        df_labeled = apply_layer1_rules_batch(df_feat, metadata, samples_per_hour=upload_samples_per_hour)

                    with st.spinner("Menjalankan prediksi AI (Weighted Ensemble)..."):
                        df_scored = score_batch_weighted(df_labeled, metadata, scaler, ocsvm, lof, clf, FEATURE_COLS)

                    st.session_state.uploaded_result = df_scored

                    st.markdown("#### 3️⃣ Hasil Preprocessing (Cleansing + Feature Engineering)")
                    cl1, cl2, cl3, cl4 = st.columns(4)
                    cl1.metric("Baris Raw", f"{cleaning_log['rows_raw']:,}")
                    cl2.metric("Setelah Time-Slice", f"{cleaning_log['rows_after_time_slice']:,}")
                    cl3.metric("Dibuang (Full-Zero)", f"{cleaning_log['rows_dropped_full_zero']:,}")
                    cl4.metric("Baris Final", f"{cleaning_log['rows_final']:,}")
                    if not cleaning_log["time_slice_applied"]:
                        st.caption("ℹ️ Time-slicing Maret–Agustus 2025 dilewati karena data upload berada di luar "
                                   "rentang tersebut (kemungkinan data live/terbaru).")
                    n_sensor_fault = int(df_labeled["molten_temp_sensor_fault_flag"].sum()) if "molten_temp_sensor_fault_flag" in df_labeled.columns else 0
                    if n_sensor_fault > 0:
                        st.caption(f"⚠️ {n_sensor_fault:,} baris memiliki `molten_temp == 0` — ditangani sebagai "
                                   "sensor fault (molten_temp_valid = NaN), bukan temperature drop asli.")
                    n_elec_fault = int(df_labeled["electrical_sensor_fault_flag"].sum()) if "electrical_sensor_fault_flag" in df_labeled.columns else 0
                    if n_elec_fault > 0:
                        st.caption(f"⚠️ {n_elec_fault:,} baris memiliki voltage_avr/current_avr/power_total "
                                   "serentak 0 — ditangani sebagai sensor fault kelistrikan, bukan heater OFF asli.")
                    n_open_circuit = int((df_labeled["reason"] == R_OPEN_CIRCUIT).sum()) if "reason" in df_labeled.columns else 0
                    if n_open_circuit > 0:
                        st.caption(f"🚨 {n_open_circuit:,} baris terindikasi **Heater Open Circuit / Broken Element** "
                                   "(current_avr ~0A padahal voltage_avr normal & heater terbaca ON).")

                    # ---- 4️⃣ Dataframe hasil label AI ----
                    st.markdown("#### 4️⃣ Dataframe Hasil Prediksi AI (Ter-label)")
                    display_cols = ["timestamp"] + CORE_FEATURES + [
                        "molten_temp_valid", "molten_temp_sensor_fault_flag",
                        "resistance", "resistance_class", "electrical_sensor_fault_flag",
                        "reason", "severity_level",
                        "is_anomaly", "xgb_anomaly_prob", "xgb_pred_severity", "ocsvm_anomaly_prob",
                        "lof_anomaly_prob", "weighted_score", "final_flag"
                    ]
                    display_cols = [c for c in display_cols if c in df_scored.columns]
                    st.dataframe(df_scored[display_cols], use_container_width=True, hide_index=True)

                    n_anom = int(df_scored["final_flag"].sum())
                    st.caption(f"Total anomali terdeteksi (Weighted Ensemble): {n_anom:,} dari "
                               f"{len(df_scored):,} baris ({n_anom/max(len(df_scored),1)*100:.2f}%)")

                    # ---- Actionable alert untuk anomali paling signifikan ----
                    if n_anom > 0:
                        worst_row = df_scored.loc[df_scored["weighted_score"].idxmax()]
                        st.markdown("##### 🚨 Alert — Anomali dengan Skor Tertinggi")
                        st.caption(f"Timestamp: {worst_row['timestamp']} · Severity: {worst_row.get('severity_level', '—')} · "
                                   f"Weighted score: {worst_row['weighted_score']:.3f}")
                        actionable_alert([worst_row.get("reason", R_NORMAL)])

                    # ---- 5️⃣ Visualisasi ----
                    st.markdown("#### 5️⃣ Visualisasi Time-Series (Tren Penuh)")
                    st.plotly_chart(
                        make_full_timeseries(df_scored, "molten_temp",
                                              "molten_temp — Tren Penuh (titik merah = anomali ensemble)",
                                              anomaly_col="final_flag", growing=False),
                        use_container_width=True,
                    )
                    vc1, vc2 = st.columns(2)
                    with vc1:
                        st.plotly_chart(make_full_timeseries(df_scored, "current_avr", "current_avr — Tren Penuh",
                                                              anomaly_col="final_flag", growing=False),
                                         use_container_width=True)
                    with vc2:
                        st.plotly_chart(make_full_timeseries(df_scored, "power_total", "power_total — Tren Penuh",
                                                              anomaly_col="final_flag", growing=False),
                                         use_container_width=True)

                    # ---- 6️⃣ Download ----
                    st.markdown("#### 6️⃣ Download Hasil")
                    st.download_button(
                        "⬇ Download Data Ter-label (CSV)",
                        data=df_scored.to_csv(index=False).encode("utf-8"),
                        file_name=f"labeled_prediction_{now_wib().strftime('%Y%m%d_%H%M%S')}.csv",
                        mime="text/csv",
                        use_container_width=True,
                    )
                except Exception as e:
                    st.error(f"Gagal memproses file: {e}")
        else:
            st.info("⬆️ Unggah file CSV/XLSX untuk memulai preprocessing otomatis & prediksi AI.")

    # ------------------------------------------------------------- TAB HISTORIS
    with tab_hist:
        # Jika ada hasil upload pada sesi ini, gunakan sebagai sumber analitik;
        # jika tidak, fallback ke data historis (labeled_data.csv) seperti biasa.
        hist_source = st.session_state.uploaded_result if st.session_state.uploaded_result is not None else df_hist
        hist_label = "data hasil upload" if st.session_state.uploaded_result is not None else "labeled_data.csv"

        if hist_source is None or len(hist_source) == 0:
            st.info("Belum ada data untuk dianalisis. Upload file pada tab pertama, atau pastikan "
                    "`labeled_data.csv` tersedia di `DATA_PATH`.")
        else:
            anomaly_col_hist = "final_flag" if "final_flag" in hist_source.columns else "is_anomaly"
            st.caption(f"Sumber: {hist_label} · {len(hist_source):,} baris · "
                       f"{int(hist_source[anomaly_col_hist].sum()):,} anomali "
                       f"({hist_source[anomaly_col_hist].mean()*100:.2f}%)")

            st.plotly_chart(make_full_timeseries(hist_source, "molten_temp",
                                                  "Timeline molten_temp — Anomali Ditandai",
                                                  anomaly_col=anomaly_col_hist),
                             use_container_width=True)

            bc1, bc2 = st.columns(2)
            with bc1:
                if "reason" in hist_source.columns:
                    st.plotly_chart(make_reason_bar(hist_source), use_container_width=True)
                else:
                    st.info("Kolom `reason` tidak tersedia pada sumber data ini.")
            with bc2:
                if "severity_level" in hist_source.columns:
                    st.plotly_chart(make_severity_bar(hist_source), use_container_width=True)
                else:
                    fig_imp = make_feature_importance(clf, FEATURE_COLS)
                    if fig_imp is not None:
                        st.plotly_chart(fig_imp, use_container_width=True)

            fig_imp = make_feature_importance(clf, FEATURE_COLS)
            if fig_imp is not None:
                st.plotly_chart(fig_imp, use_container_width=True)

    # ------------------------------------------------------------------ TAB DEBUG
    with tab_debug:
        st.markdown("#### Model Metadata")
        st.json(metadata)

        st.markdown("#### Formula Weighted Ensemble")
        st.markdown(f"""
        <div class="card">
            <p>weighted_score = {W_RULE:.2f} × rule_flag + {W_XGB:.2f} × P(anomaly | XGBoost multi-class) +
               {W_OCSVM:.2f} × P(OCSVM) + {W_LOF:.2f} × P(LOF)</p>
            <p>Threshold: <b>weighted_score ≥ {SCORE_THRESHOLD:.2f}</b> → Anomali
               (rule_flag = 1 tetap men-trigger Anomali sebagai safety hard-override).</p>
            <p>severity_level ∈ {{NORMAL, WARNING, CRITICAL, FATAL/EMERGENCY}} dihasilkan lewat
               np.select atas 12 rule fisik Layer-1 (prioritas: {" > ".join(REASON_ORDER)}).</p>
        </div>
        """, unsafe_allow_html=True)

        if st.session_state.uploaded_result is not None:
            st.markdown("#### Reference Stats (dari data hasil upload sesi ini)")
            st.json(compute_reference_stats(st.session_state.uploaded_result))
        else:
            st.markdown("#### Reference Stats (dari data historis)")
            st.json(ref_stats if ref_stats else {"info": "Tidak ada data historis dimuat."})

    st.caption("Mode Upload menjalankan pipeline lengkap secara otomatis di backend: cleansing "
               "(Notebook 1) → feature engineering (molten_temp_valid) + severity-based rule labeling "
               "(Notebook 2) → Weighted Ensemble Rule(40%) + XGBoost multi-class(40%) + OCSVM(10%) + LOF(10%) "
               "(Notebook 3).")

Overwriting dashboard.py


In [12]:
import subprocess, time
from pyngrok import ngrok

# ⬇️ TARO TOKEN NGROK KAMU DI SINI ⬇️
NGROK_TOKEN = "3Gnk8JxXPBDDGIx8MuKUNQmXovV_71x1nAG3472LpXRXqL3BR"

ngrok.set_auth_token(NGROK_TOKEN)

# Jalankan Streamlit sebagai proses background
process = subprocess.Popen([
    "streamlit", "run", "dashboard.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
])

print("Menunggu Streamlit menyala...")
time.sleep(8)

# Tutup tunnel lama kalau ada (jaga-jaga saat re-run cell ini)
ngrok.kill()

public_url = ngrok.connect(8501, "http")
print("✅ Dashboard tersedia di:", public_url)

Menunggu Streamlit menyala...


2026-08-04 01:21:27.884 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            


2026-08-04 01:21:29.259 Port 8501 is not available


✅ Dashboard tersedia di: NgrokTunnel: "https://t-shirt-velvet-stonewall.ngrok-free.dev" -> "http://localhost:8501"


2026-08-04 01:22:05.771 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-08-04 01:22:05.778 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-08-04 01:22:05.785 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-08-04 01:22:05.850 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-08-04 01:22:05.956 